# Radar Financeiro Massivo V9 R4 R2 — execução completa de teste

Versão completa para execução longa e diagnóstico do pipeline refatorado.

Princípios desta versão:

- Q1 e Q5 usam a partição física `NR_PTC` da origem transacional;
- Q2 usa leitura JDBC paralela pelo líder de índice `CD_UOR_CC` dentro do intervalo relevante do público;
- Q3 usa a réplica Hive `DB2DFE.REN_AVLD_PF`;
- Q4 percorre `DT_REF` em ordem decrescente e lê snapshots distribuídos, sem coletar clientes no driver;
- fronteiras críticas (`radar_publico`, `radar_contexto`, `radar_movimentos`, `radar_efetivos`) são congeladas com `localCheckpoint(eager=True)` para impedir recomputação JDBC;
- cada etapa pesada possui `%%time`, ação real e saída de diagnóstico;
- o público permanece como base de cardinalidade até `radar_resultado_final`;
- reconciliação, classificação, agregação, motor, 142 colunas e DDL permanecem iguais à baseline SQL funcional;
- publicação é opcional durante o teste. Por padrão `PUBLICAR_RESULTADO = False`.

Execute as células em ordem. Se uma etapa falhar, os tempos e volumes das etapas anteriores permanecem impressos para diagnóstico.


## 1. Ambiente e parâmetros


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao="radar-massivo-v9-sprint1",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_local.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        num_executors=6,
        executor_memory="4g",
        executor_cores=4,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "2g",

            "spark.dynamicAllocation.enabled": "true",
            "spark.dynamicAllocation.minExecutors": "4",
            "spark.dynamicAllocation.initialExecutors": "6",
            "spark.dynamicAllocation.maxExecutors": "8",

            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",

            "spark.speculation": "false",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64m",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "320",

            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

    print("[RADAR V9] Sessão Spark inicializada.")

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


In [ ]:
%%time
%%spark
import os
import time

periodo = 1
FETCHSIZE = 10000
QUERY_TIMEOUT = 900

# Origem transacional: NR_PTC físico 1..100.
NR_PTC_LOWER_BOUND = 1
NR_PTC_UPPER_BOUND = 101
Q1_JDBC_PARTITIONS = 4
Q5_JDBC_PARTITIONS = 4

# Q2 é muito menor que Q1/Q4/Q5; começa conservador.
Q2_JDBC_PARTITIONS = 4

# Q4 R4: uma única consulta DB2 reduzida pelo público e pela maior DT_REF.
# Não particionar a consulta pesada em CD_CLI para não repetir o subplano no DB2.
Q4_JDBC_PARTITIONS = 1
Q4_QUERY_TIMEOUT = 3600

# Teste completo calcula tudo, mas não publica por padrão.
# Alterar para True somente quando quiser testar também DROP -> DDL -> INSERT -> readback.
PUBLICAR_RESULTADO = False

if type(periodo) is not int or not 1 <= periodo <= 6:
    raise ValueError('periodo deve ser inteiro entre 1 e 6.')

DATA_EXECUCAO = str(obter_variavel_ambiente('HOJE'))[:10]
conector_db2 = criar_conector_db2_spark(env=dict(os.environ))
DB_ATIVO = spark.catalog.currentDatabase()
metricas_execucao = {}

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW radar_parametros AS
SELECT
    DATE('{DATA_EXECUCAO}') AS DT_EXEA,
    TRUNC(DATE('{DATA_EXECUCAO}'), 'MM') AS DT_MES_EXEA,
    {periodo} AS PERIODO,
    DATE_ADD(
        TRUNC(ADD_MONTHS(DATE('{DATA_EXECUCAO}'), -1), 'MM'),
        LEAST(
            DAY(DATE('{DATA_EXECUCAO}')),
            DAY(LAST_DAY(ADD_MONTHS(DATE('{DATA_EXECUCAO}'), -1)))
        ) - 1
    ) AS DT_INI_PUBLICO
""")

print('Database ativo:', DB_ATIVO)
print('Data execução:', DATA_EXECUCAO, '| periodo:', periodo)
print('Spark:', spark.version)
print('Q1/Q5 JDBC partitions:', Q1_JDBC_PARTITIONS, '/', Q5_JDBC_PARTITIONS)
print('Q4 JDBC partitions:', Q4_JDBC_PARTITIONS)
print('Q4 query timeout:', Q4_QUERY_TIMEOUT)
print('Publicar resultado:', PUBLICAR_RESULTADO)


## 2. Q1 — Público

### Q1.1 — leitura física + pré-agregação no DB2

A consulta é agregada por `NR_PTC + CD_CLI` ainda no DB2. O Spark JDBC divide a leitura por `NR_PTC`, evitando transportar as transações brutas do público.


In [ ]:
%%time
%%spark
# Q1.1 — reduzir a origem no DB2 por partição física + cliente.
t0_etapa = time.perf_counter()

p = spark.sql("""
SELECT DT_EXEA, DT_INI_PUBLICO
FROM radar_parametros
""").first()

sql_q1_particionado = f"""
SELECT
    NR_PTC,
    CD_CLI,

    MAX(TS_INCL_TRAN) AS TS_INCL_TRAN_REF_PART,

    MIN(NR_CPF_CNPJ_TITR) AS CPF_MIN_PART,
    MAX(NR_CPF_CNPJ_TITR) AS CPF_MAX_PART,

    MIN(
        CASE
            WHEN NR_MCA_PCT_OPB = 999999999
             AND CD_PRD = 6
             AND NR_AG_TITR IS NOT NULL
             AND CD_CT_TITR IS NOT NULL
             AND TRIM(CAST(CD_CT_TITR AS VARCHAR(50))) <> ''
            THEN NR_AG_TITR
        END
    ) AS AG_MIN_PART,

    MAX(
        CASE
            WHEN NR_MCA_PCT_OPB = 999999999
             AND CD_PRD = 6
             AND NR_AG_TITR IS NOT NULL
             AND CD_CT_TITR IS NOT NULL
             AND TRIM(CAST(CD_CT_TITR AS VARCHAR(50))) <> ''
            THEN NR_AG_TITR
        END
    ) AS AG_MAX_PART,

    MIN(
        CASE
            WHEN NR_MCA_PCT_OPB = 999999999
             AND CD_PRD = 6
             AND NR_AG_TITR IS NOT NULL
             AND CD_CT_TITR IS NOT NULL
             AND TRIM(CAST(CD_CT_TITR AS VARCHAR(50))) <> ''
            THEN CD_CT_TITR
        END
    ) AS CC_MIN_PART,

    MAX(
        CASE
            WHEN NR_MCA_PCT_OPB = 999999999
             AND CD_PRD = 6
             AND NR_AG_TITR IS NOT NULL
             AND CD_CT_TITR IS NOT NULL
             AND TRIM(CAST(CD_CT_TITR AS VARCHAR(50))) <> ''
            THEN CD_CT_TITR
        END
    ) AS CC_MAX_PART

FROM DB2GFP.TRAN_RLZD_INST_PCT
WHERE CD_CLI IS NOT NULL
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TS_INCL_TRAN >= TIMESTAMP('{p["DT_INI_PUBLICO"]} 00:00:00')
  AND TS_INCL_TRAN < TIMESTAMP('{p["DT_EXEA"]} 00:00:00')
GROUP BY
    NR_PTC,
    CD_CLI
"""

df_q1_parcial = conector_db2.sql(
    sql_q1_particionado,
    fetchsize=FETCHSIZE,
    query_timeout=QUERY_TIMEOUT,
    partition_column="NR_PTC",
    lower_bound=NR_PTC_LOWER_BOUND,
    upper_bound=NR_PTC_UPPER_BOUND,
    num_partitions=Q1_JDBC_PARTITIONS,
)

df_q1_parcial.createOrReplaceTempView("radar_q1_parcial")

spark.sql("CACHE TABLE radar_q1_parcial")

qt_q1_parcial = spark.sql("""
SELECT COUNT(*) AS N
FROM radar_q1_parcial
""").first()["N"]

print("Q1 parcial materializada:", qt_q1_parcial, "linhas NR_PTC+CD_CLI")
metricas_execucao['Q1_LEITURA'] = {'segundos': round(time.perf_counter()-t0_etapa,3), 'linhas_parciais': int(qt_q1_parcial)}
print('[TEMPO Q1_LEITURA]', metricas_execucao['Q1_LEITURA'])


### Q1.2 — consolidação global no grão de cliente


In [ ]:
%%time
%%spark
# Q1.2 — consolidar os resultados parciais no grão de cliente.
t0_etapa = time.perf_counter()

spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_q1_agregado AS
SELECT
    CAST(CD_CLI AS INT) AS CD_CLI,

    MAX(TS_INCL_TRAN_REF_PART) AS TS_INCL_TRAN_REF,

    MIN(CPF_MIN_PART) AS CPF_MIN,
    MAX(CPF_MAX_PART) AS CPF_MAX,

    MIN(AG_MIN_PART) AS AG_MIN,
    MAX(AG_MAX_PART) AS AG_MAX,

    MIN(CC_MIN_PART) AS CC_MIN,
    MAX(CC_MAX_PART) AS CC_MAX

FROM radar_q1_parcial
GROUP BY CD_CLI
""")

spark.sql(r"""
CREATE OR REPLACE TEMP VIEW radar_publico AS
WITH regras AS (
    SELECT
        CD_CLI,
        TS_INCL_TRAN_REF,

        CASE
            WHEN CPF_MIN IS NOT NULL
             AND CPF_MIN = CPF_MAX
            THEN 'S'
            ELSE 'N'
        END AS FL_CPF_UNICO,

        CASE
            WHEN CPF_MIN IS NOT NULL
             AND CPF_MIN = CPF_MAX
            THEN CAST(CPF_MIN AS DECIMAL(14,0))
        END AS CD_CPF,

        CASE
            WHEN AG_MIN IS NOT NULL
             AND CC_MIN IS NOT NULL
             AND AG_MIN = AG_MAX
             AND CC_MIN = CC_MAX
            THEN 'S'
            ELSE 'N'
        END AS FL_CONTA_ELEGIVEL_UNICA,

        CASE
            WHEN AG_MIN IS NOT NULL
             AND CC_MIN IS NOT NULL
             AND AG_MIN = AG_MAX
             AND CC_MIN = CC_MAX
            THEN TRIM(CAST(AG_MIN AS STRING))
        END AS AG_TXT,

        CASE
            WHEN AG_MIN IS NOT NULL
             AND CC_MIN IS NOT NULL
             AND AG_MIN = AG_MAX
             AND CC_MIN = CC_MAX
            THEN TRIM(CAST(CC_MIN AS STRING))
        END AS CC_TXT

    FROM radar_q1_agregado
),
normalizacao AS (
    SELECT
        *,
        CASE
            WHEN REGEXP_REPLACE(CC_TXT, '^0+', '') = '' THEN '0'
            ELSE REGEXP_REPLACE(CC_TXT, '^0+', '')
        END AS CC_SIG
    FROM regras
)
SELECT
    CAST(CD_CLI AS INT) AS CD_CLI,
    TS_INCL_TRAN_REF,
    FL_CPF_UNICO,
    CD_CPF,
    FL_CONTA_ELEGIVEL_UNICA,

    CASE
        WHEN FL_CONTA_ELEGIVEL_UNICA = 'S'
         AND AG_TXT RLIKE '^[0-9]+$'
         AND CC_TXT RLIKE '^[0-9]+$'
         AND CAST(AG_TXT AS BIGINT) BETWEEN -2147483648 AND 2147483647
         AND LENGTH(CC_SIG) <= 11
        THEN CAST(AG_TXT AS INT)
    END AS CD_UOR_CC_NORM,

    CASE
        WHEN FL_CONTA_ELEGIVEL_UNICA = 'S'
         AND AG_TXT RLIKE '^[0-9]+$'
         AND CC_TXT RLIKE '^[0-9]+$'
         AND CAST(AG_TXT AS BIGINT) BETWEEN -2147483648 AND 2147483647
         AND LENGTH(CC_SIG) <= 11
        THEN CAST(CC_SIG AS DECIMAL(11,0))
    END AS NR_CC_NORM

FROM normalizacao
""")

# Congelar o público como snapshot lógico independente da leitura JDBC.
# Isso evita que etapas posteriores recomputem Q1 caso blocos de cache sejam invalidados/evictos.
df_publico_snapshot = spark.table("radar_publico").localCheckpoint(eager=True)
df_publico_snapshot.createOrReplaceTempView("radar_publico")

qt_publico = df_publico_snapshot.count()

print("Q1 público congelado:", qt_publico, "clientes")
metricas_execucao['Q1_CONSOLIDACAO'] = {'segundos': round(time.perf_counter()-t0_etapa,3), 'clientes': int(qt_publico)}
print('[TEMPO Q1_CONSOLIDACAO]', metricas_execucao['Q1_CONSOLIDACAO'])


### Q1.3 — gate do público


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
metricas_q1 = spark.sql("""
SELECT
    COUNT(*) AS QT_LINHAS,
    COUNT(DISTINCT CD_CLI) AS QT_CLIENTES,
    SUM(CASE WHEN CD_CLI IS NULL THEN 1 ELSE 0 END) AS QT_CD_CLI_NULO
FROM radar_publico
""").first()

qt_linhas = int(metricas_q1["QT_LINHAS"])
qt_clientes = int(metricas_q1["QT_CLIENTES"])
qt_nulos = int(metricas_q1["QT_CD_CLI_NULO"] or 0)

if qt_linhas == 0:
    raise RuntimeError("Q1 falhou: público vazio.")

if qt_nulos != 0:
    raise RuntimeError(
        f"Q1 falhou: existem {qt_nulos} linhas com CD_CLI nulo."
    )

if qt_linhas != qt_clientes:
    raise RuntimeError(
        "Q1 falhou: radar_publico não está no grão de uma linha por CD_CLI. "
        f"linhas={qt_linhas}, clientes_distintos={qt_clientes}."
    )

spark.sql("UNCACHE TABLE radar_q1_parcial")

print("[OK] Sprint 1")
print("Público:", qt_linhas, "clientes")
print("Grão: 1 linha por CD_CLI")
print("radar_q1_parcial: cache liberado")
print("radar_publico: snapshot congelado para as próximas etapas")
metricas_execucao['Q1_GATE'] = {'segundos': round(time.perf_counter()-t0_etapa,3), 'clientes': qt_linhas}
print('[TEMPO Q1_GATE]', metricas_execucao['Q1_GATE'])


## 3. Q2 — Ciclo financeiro

A fonte tem escala bem menor. A leitura continua simples, mas passa a ser paralela por `CD_UOR_CC`, líder dos índices da tabela, e limitada ao intervalo de agências realmente presente nas contas normalizadas do público.


In [ ]:
%%time
%%spark
# Q2 — leitura distribuída + seleção do ciclo mais recente + fallback.
t0_etapa = time.perf_counter()

q2_publico_entrada = spark.sql("""
SELECT COUNT(*) N, COUNT(DISTINCT CD_CLI) U
FROM radar_publico
""").first()
if q2_publico_entrada['N'] != qt_publico or q2_publico_entrada['N'] != q2_publico_entrada['U']:
    raise RuntimeError(f'Q2 recebeu público instável: {q2_publico_entrada}, esperado={qt_publico}')
print('[Q2 ENTRADA] público estável:', q2_publico_entrada['N'])

b_q2 = spark.sql("""
SELECT
    MIN(CD_UOR_CC_NORM) AS AG_MIN,
    MAX(CD_UOR_CC_NORM) AS AG_MAX,
    COUNT(*) AS CONTAS_VALIDAS
FROM radar_publico
WHERE CD_UOR_CC_NORM IS NOT NULL
  AND NR_CC_NORM IS NOT NULL
""").first()

if (b_q2['CONTAS_VALIDAS'] or 0) == 0:
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW radar_q2_raw AS
    SELECT
        CAST(NULL AS INT) AS CD_UOR_CC,
        CAST(NULL AS DECIMAL(11,0)) AS NR_CC,
        CAST(NULL AS SMALLINT) AS DD_INC_MM_CLC_BLC,
        CAST(NULL AS TIMESTAMP) AS TS_ULT_EXEA_PSQ
    WHERE 1=0
    """)
else:
    ag_min = int(b_q2['AG_MIN'])
    ag_max = int(b_q2['AG_MAX'])
    df_q2 = conector_db2.sql(f"""
    SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
    FROM DB2GFP.CT_GRDR_FNCO
    WHERE CD_UOR_CC BETWEEN {ag_min} AND {ag_max}
    """,
        fetchsize=FETCHSIZE,
        query_timeout=QUERY_TIMEOUT,
        partition_column='CD_UOR_CC',
        lower_bound=ag_min,
        upper_bound=ag_max + 1,
        num_partitions=Q2_JDBC_PARTITIONS,
    )
    df_q2.createOrReplaceTempView('radar_q2_raw')

spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_contexto_ciclo AS
WITH contas AS (
    SELECT DISTINCT
        CD_UOR_CC_NORM AS CD_UOR_CC,
        NR_CC_NORM AS NR_CC
    FROM radar_publico
    WHERE CD_UOR_CC_NORM IS NOT NULL
      AND NR_CC_NORM IS NOT NULL
), r AS (
    SELECT
        c.CD_UOR_CC,
        c.NR_CC,
        c.DD_INC_MM_CLC_BLC,
        c.TS_ULT_EXEA_PSQ,
        ROW_NUMBER() OVER (
            PARTITION BY c.CD_UOR_CC, c.NR_CC
            ORDER BY c.TS_ULT_EXEA_PSQ DESC
        ) AS RN
    FROM radar_q2_raw c
    INNER JOIN contas p
        ON c.CD_UOR_CC = p.CD_UOR_CC
       AND c.NR_CC = p.NR_CC
), ciclo AS (
    SELECT
        CD_UOR_CC AS CD_UOR_CC_NORM,
        NR_CC AS NR_CC_NORM,
        DD_INC_MM_CLC_BLC,
        TS_ULT_EXEA_PSQ AS TS_DD_INC_MM_CLC_BLC_REF
    FROM r
    WHERE RN = 1
)
SELECT
    p.*,
    c.TS_DD_INC_MM_CLC_BLC_REF,
    CAST(c.DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC,
    CAST(
        CASE
            WHEN p.CD_UOR_CC_NORM IS NULL THEN NULL
            WHEN c.DD_INC_MM_CLC_BLC IS NULL THEN 1
            ELSE c.DD_INC_MM_CLC_BLC
        END AS SMALLINT
    ) AS DD_INC_MM_CLC_BLC_FALLBACK
FROM radar_publico p
LEFT JOIN ciclo c
    ON p.CD_UOR_CC_NORM = c.CD_UOR_CC_NORM
   AND p.NR_CC_NORM = c.NR_CC_NORM
""")

spark.sql('CACHE TABLE radar_contexto_ciclo')
q2_gate = spark.sql("""
SELECT
    COUNT(*) AS N,
    COUNT(DISTINCT CD_CLI) AS U,
    SUM(
        CASE
            WHEN DD_INC_MM_CLC_BLC_FALLBACK IS NOT NULL
             AND DD_INC_MM_CLC_BLC_FALLBACK NOT BETWEEN 1 AND 31
            THEN 1 ELSE 0
        END
    ) AS INVALIDOS
FROM radar_contexto_ciclo
""").first()

if q2_gate['N'] != qt_publico or q2_gate['N'] != q2_gate['U']:
    raise RuntimeError(f'Q2 alterou o grão do público: {q2_gate}')
if (q2_gate['INVALIDOS'] or 0) > 0:
    raise RuntimeError(f"Dia de ciclo inválido em {q2_gate['INVALIDOS']} cliente(s).")

metricas_execucao['Q2'] = {
    'segundos': round(time.perf_counter()-t0_etapa,3),
    'clientes': int(q2_gate['N']),
    'contas_validas_publico': int(b_q2['CONTAS_VALIDAS'] or 0),
}
print('[Q2]', metricas_execucao['Q2'])


## 4. Q3 — Renda

A renda é resolvida na réplica Hive `DB2DFE.REN_AVLD_PF`. A etapa lê somente as colunas necessárias, restringe pelos CPFs únicos do público e materializa diretamente o contexto enriquecido.


In [ ]:
%%time
%%spark
# Q3 — renda mais recente por CPF único na réplica Hive.
t0_etapa = time.perf_counter()

q3_contexto_entrada = spark.sql("""
SELECT COUNT(*) N, COUNT(DISTINCT CD_CLI) U
FROM radar_contexto_ciclo
""").first()
if q3_contexto_entrada['N'] != qt_publico or q3_contexto_entrada['N'] != q3_contexto_entrada['U']:
    raise RuntimeError(f'Q3 recebeu contexto de ciclo instável: {q3_contexto_entrada}, esperado={qt_publico}')
print('[Q3 ENTRADA] contexto ciclo estável:', q3_contexto_entrada['N'])

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW radar_contexto_renda AS
WITH cpf AS (
    SELECT DISTINCT CD_CPF AS NR_CPF_BASE_SRF
    FROM radar_publico
    WHERE FL_CPF_UNICO='S'
      AND CD_CPF IS NOT NULL
), r AS (
    SELECT
        h.NR_CPF_BASE_SRF,
        h.DT_INCL_REN_AVLD,
        h.VL_REN,
        ROW_NUMBER() OVER (
            PARTITION BY h.NR_CPF_BASE_SRF
            ORDER BY h.DT_INCL_REN_AVLD DESC
        ) AS RN
    FROM DB2DFE.REN_AVLD_PF h
    INNER JOIN cpf p
        ON h.NR_CPF_BASE_SRF = p.NR_CPF_BASE_SRF
)
SELECT
    c.*,
    CAST(r.DT_INCL_REN_AVLD AS DATE) AS DT_REN_PRES_REF,
    CAST(r.VL_REN * {periodo} AS DECIMAL(17,2)) AS VL_REN_PRES
FROM radar_contexto_ciclo c
LEFT JOIN r
    ON c.CD_CPF = CAST(r.NR_CPF_BASE_SRF AS DECIMAL(14,0))
   AND r.RN = 1
""")

spark.sql('CACHE TABLE radar_contexto_renda')
q3_gate = spark.sql("""
SELECT
    COUNT(*) AS N,
    COUNT(DISTINCT CD_CLI) AS U,
    SUM(CASE WHEN DT_REN_PRES_REF IS NOT NULL THEN 1 ELSE 0 END) AS COM_RENDA
FROM radar_contexto_renda
""").first()

if q3_gate['N'] != qt_publico or q3_gate['N'] != q3_gate['U']:
    raise RuntimeError(f'Q3 alterou o grão do público: {q3_gate}')

metricas_execucao['Q3'] = {
    'segundos': round(time.perf_counter()-t0_etapa,3),
    'clientes': int(q3_gate['N']),
    'clientes_com_renda': int(q3_gate['COM_RENDA'] or 0),
}
print('[Q3]', metricas_execucao['Q3'])


## 5. Q4 — Perfil financeiro

A Q4 foi completamente redesenhada em torno da estrutura física da fonte.

1. Busca somente a lista pequena de `DT_REF <= DT_EXEA`.
2. Percorre as referências da mais recente para a mais antiga.
3. Em cada referência, o DB2 usa `DT_REF` fixa e leitura JDBC particionada por `CD_CLI`.
4. O Spark relaciona o snapshot apenas aos clientes ainda pendentes.
5. Um cliente resolvido não participa das referências anteriores.
6. Mais de uma linha para o mesmo cliente na referência escolhida continua sendo erro bloqueante.

Nenhuma lista massiva de clientes é coletada para o driver.


In [ ]:
%%time
%%spark
# Q4.1 R4 — Público -> maior DT_REF -> atributos do perfil.
#
# Regra funcional preservada da V25:
#   1) considerar somente CD_CLI do público;
#   2) considerar somente DT_REF <= DATA_EXECUCAO;
#   3) selecionar a maior DT_REF de cada CD_CLI;
#   4) trazer todas as linhas existentes nessa maior DT_REF;
#   5) se houver mais de uma linha na maior DT_REF, o gate interrompe.
#
# Estratégia R4:
#   - reconstruir no DB2 apenas a PERTINÊNCIA AO PÚBLICO usando os mesmos
#     filtros-base da Q1;
#   - juntar esse conjunto reduzido à tabela de perfil;
#   - calcular MAX(DT_REF) por CD_CLI no DB2;
#   - voltar à DVS_GRDR_FNCO_PF usando CD_CLI + DT_REF_MAX;
#   - somente o resultado reduzido atravessa JDBC;
#   - radar_publico congelado continua sendo a autoridade final no Spark.

t0_q4 = time.perf_counter()

p_q4 = spark.sql("""
SELECT DT_EXEA, DT_INI_PUBLICO
FROM radar_parametros
""").first()

sql_q4_r4 = f"""
SELECT
    D.CD_CLI,
    D.DT_REF,
    D.CD_MAC_PRFL_CLI,
    D.NM_MAC_PRFL_CLI,
    D.CD_MIC_PRFL_CLI,
    D.NM_MIC_PRFL_CLI
FROM DB2D1D.DVS_GRDR_FNCO_PF D
INNER JOIN (
    SELECT
        H.CD_CLI,
        MAX(H.DT_REF) AS DT_REF_MAX
    FROM DB2D1D.DVS_GRDR_FNCO_PF H
    INNER JOIN (
        SELECT
            P1.CD_CLI
        FROM (
            SELECT
                P.NR_PTC,
                P.CD_CLI
            FROM DB2GFP.TRAN_RLZD_INST_PCT P
            WHERE P.CD_CLI IS NOT NULL
              AND P.CD_EST_TRAN_INST = 0
              AND P.CD_TIP_PSS = 1
              AND P.TS_INCL_TRAN >= TIMESTAMP('{p_q4["DT_INI_PUBLICO"]} 00:00:00')
              AND P.TS_INCL_TRAN < TIMESTAMP('{p_q4["DT_EXEA"]} 00:00:00')
            GROUP BY
                P.NR_PTC,
                P.CD_CLI
        ) P1
        GROUP BY
            P1.CD_CLI
    ) PUB
        ON PUB.CD_CLI = H.CD_CLI
    WHERE H.DT_REF <= DATE('{p_q4["DT_EXEA"]}')
    GROUP BY
        H.CD_CLI
) M
    ON D.CD_CLI = M.CD_CLI
   AND D.DT_REF = M.DT_REF_MAX
"""

print("[Q4 R4] Iniciando consulta DB2: público -> MAX(DT_REF) -> perfil.")

# Uma única execução JDBC.
# Não usar partition_column aqui: dividir a saída poderia fazer o DB2
# repetir o mesmo subplano pesado em várias consultas independentes.
df_q4_db2 = conector_db2.sql(
    sql_q4_r4,
    fetchsize=FETCHSIZE,
    query_timeout=Q4_QUERY_TIMEOUT,
)

# Materialização explícita: depois daqui, nenhuma action posterior reabre a Q4 no DB2.
df_q4_db2_snapshot = df_q4_db2.localCheckpoint(eager=True)
df_q4_db2_snapshot.createOrReplaceTempView("radar_q4_db2")

m_q4_db2 = spark.sql("""
SELECT
    COUNT(*) AS LINHAS,
    COUNT(DISTINCT CD_CLI) AS CLIENTES,
    MAX(DT_REF) AS DT_REF_MAX_RETORNADA
FROM radar_q4_db2
""").first()

print(
    "[Q4 R4 DB2]",
    {
        "linhas": int(m_q4_db2["LINHAS"]),
        "clientes": int(m_q4_db2["CLIENTES"]),
        "dt_ref_max_retornada": str(m_q4_db2["DT_REF_MAX_RETORNADA"])
            if m_q4_db2["DT_REF_MAX_RETORNADA"] is not None else None,
    },
)

# A Q1 congelada é a autoridade sobre quem pertence à execução.
# Isso também elimina eventual cliente que tenha entrado na origem DB2
# depois do snapshot lógico criado pela Q1.
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_perfil AS
SELECT
    CAST(d.CD_CLI AS INT) AS CD_CLI,
    CAST(d.DT_REF AS DATE) AS DT_REF_PRFL,
    CAST(d.CD_MAC_PRFL_CLI AS INT) AS CD_MAC_PRFL_CLI,
    d.NM_MAC_PRFL_CLI,
    CAST(d.CD_MIC_PRFL_CLI AS INT) AS CD_MIC_PRFL_CLI,
    d.NM_MIC_PRFL_CLI
FROM radar_q4_db2 d
INNER JOIN radar_publico p
    ON d.CD_CLI = p.CD_CLI
""")

# O gate abaixo materializa radar_perfil e prova a regra de empate.
spark.sql("CACHE TABLE radar_perfil")

q4_perfil_gate = spark.sql(f"""
SELECT
    COUNT(*) AS N,
    COUNT(DISTINCT CD_CLI) AS U,
    SUM(
        CASE
            WHEN DT_REF_PRFL > DATE('{p_q4["DT_EXEA"]}')
            THEN 1 ELSE 0
        END
    ) AS POSTERIORES
FROM radar_perfil
""").first()

if (q4_perfil_gate["POSTERIORES"] or 0) > 0:
    raise RuntimeError(
        "Q4 retornou perfil posterior à DATA_EXECUCAO para "
        f"{q4_perfil_gate['POSTERIORES']} linha(s)."
    )

# A V25 interrompia quando havia mais de uma linha na MAIOR DT_REF.
# Como a consulta DB2 retorna todas as linhas da DT_REF_MAX,
# N > U representa exatamente essa ambiguidade funcional.
if q4_perfil_gate["N"] != q4_perfil_gate["U"]:
    raise RuntimeError(
        "Q4 retornou mais de uma linha na maior DT_REF elegível para "
        f"{int(q4_perfil_gate['N']) - int(q4_perfil_gate['U'])} "
        "linha(s) excedente(s)."
    )

q4_clientes_com_perfil = int(q4_perfil_gate["U"])
q4_clientes_sem_perfil = int(qt_publico) - q4_clientes_com_perfil

# Compatibilidade com células posteriores da R3.
q4_views_encontrados = []

metricas_execucao["Q4_LEITURA"] = {
    "segundos": round(time.perf_counter() - t0_q4, 3),
    "estrategia": "PUBLICO_DB2_MAX_DT_REF",
    "linhas_db2": int(m_q4_db2["LINHAS"]),
    "clientes_db2": int(m_q4_db2["CLIENTES"]),
    "clientes_com_perfil": q4_clientes_com_perfil,
    "clientes_sem_perfil": q4_clientes_sem_perfil,
}

print("[Q4 R4]", metricas_execucao["Q4_LEITURA"])

# radar_perfil já está materializado pelo gate.
# O snapshot JDBC bruto pode ser liberado.
try:
    df_q4_db2_snapshot.unpersist(blocking=False)
except Exception:
    pass


In [ ]:
%%time
%%spark
# Q4.2 — incorporar perfil ao contexto sem eliminar clientes.
t0_etapa = time.perf_counter()

spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_contexto_perfil AS
SELECT
    c.*,
    p.DT_REF_PRFL,
    p.CD_MAC_PRFL_CLI,
    p.NM_MAC_PRFL_CLI,
    p.CD_MIC_PRFL_CLI,
    p.NM_MIC_PRFL_CLI
FROM radar_contexto_renda c
LEFT JOIN radar_perfil p
    ON c.CD_CLI = p.CD_CLI
""")

spark.sql('CACHE TABLE radar_contexto_perfil')
q4_contexto_gate = spark.sql("""
SELECT COUNT(*) N, COUNT(DISTINCT CD_CLI) U,
       SUM(CASE WHEN DT_REF_PRFL IS NOT NULL THEN 1 ELSE 0 END) COM_PERFIL
FROM radar_contexto_perfil
""").first()

if q4_contexto_gate['N'] != qt_publico or q4_contexto_gate['N'] != q4_contexto_gate['U']:
    raise RuntimeError(f'Q4 alterou o grão do público: {q4_contexto_gate}')

metricas_execucao['Q4_CONTEXTO'] = {
    'segundos': round(time.perf_counter()-t0_etapa,3),
    'clientes': int(q4_contexto_gate['N']),
    'clientes_com_perfil': int(q4_contexto_gate['COM_PERFIL'] or 0),
}
print('[Q4 CONTEXTO]', metricas_execucao['Q4_CONTEXTO'])


## 6. Janela financeira → `radar_contexto`


In [ ]:
%%time
%%spark
# Janela — Calcular os ciclos financeiros fechados por cliente.
t0_etapa = time.perf_counter()
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_contexto AS
WITH p AS (SELECT * FROM radar_parametros), a AS (
    SELECT c.*, p.DT_EXEA, p.DT_MES_EXEA, p.PERIODO,
           TRUNC(TO_DATE(c.TS_INCL_TRAN_REF), 'MM') REF_MES,
           CAST(c.DD_INC_MM_CLC_BLC_FALLBACK AS INT) DIA
    FROM radar_contexto_perfil c CROSS JOIN p
), b AS (
    SELECT *, DATE_ADD(REF_MES, LEAST(DIA, DAY(LAST_DAY(REF_MES)))-1) CANDIDATO,
              ADD_MONTHS(REF_MES,-1) REF_MES_ANT
    FROM a
), c AS (
    SELECT *, CASE WHEN TO_TIMESTAMP(TS_INCL_TRAN_REF)>=TO_TIMESTAMP(CANDIDATO) THEN CANDIDATO
                   ELSE DATE_ADD(REF_MES_ANT, LEAST(DIA,DAY(LAST_DAY(REF_MES_ANT)))-1) END INICIO_ABERTO
    FROM b
), d AS (
    SELECT *, DATE_SUB(INICIO_ABERTO,1) DT_FIM_CALC,
              ADD_MONTHS(TRUNC(INICIO_ABERTO,'MM'),-PERIODO) MES_INI
    FROM c
)
SELECT
    CD_CLI, DT_EXEA, DT_MES_EXEA, TS_INCL_TRAN_REF, FL_CPF_UNICO, CD_CPF,
    FL_CONTA_ELEGIVEL_UNICA, TS_DD_INC_MM_CLC_BLC_REF, DD_INC_MM_CLC_BLC,
    DD_INC_MM_CLC_BLC_FALLBACK, DT_REN_PRES_REF, VL_REN_PRES, DT_REF_PRFL,
    CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI,
    CASE WHEN DIA IS NULL THEN CAST(NULL AS DATE) ELSE DATE_ADD(MES_INI,LEAST(DIA,DAY(LAST_DAY(MES_INI)))-1) END DT_REF_INI,
    CASE WHEN DIA IS NULL THEN CAST(NULL AS DATE) ELSE DT_FIM_CALC END DT_REF_FIM
FROM d
""")
# Fronteira estável: cortar a linhagem Q1/Q2/Q3/Q4 antes de entrar em Q5.
df_contexto_snapshot = spark.table('radar_contexto').localCheckpoint(eager=True)
df_contexto_snapshot.createOrReplaceTempView('radar_contexto')

g=spark.sql('SELECT COUNT(*) N, COUNT(DISTINCT CD_CLI) U FROM radar_contexto').first()
if g['N']!=qt_publico or g['N']!=g['U']:
    raise RuntimeError(f'Contexto alterou o público: {g}, esperado={qt_publico}')

# Agora as dependências anteriores podem ser liberadas sem reconstruir o contexto.
for tabela in ('radar_contexto_ciclo','radar_contexto_renda','radar_perfil','radar_contexto_perfil'):
    try:
        spark.sql(f'UNCACHE TABLE {tabela}')
    except Exception:
        pass
for v in q4_views_encontrados:
    try:
        spark.sql(f'UNCACHE TABLE {v}')
    except Exception:
        pass

print('Contexto congelado:', g['N'], 'clientes')
metricas_execucao['JANELA_CONTEXTO'] = {'segundos': round(time.perf_counter()-t0_etapa,3), 'clientes': int(g['N'])}
print('[JANELA]', metricas_execucao['JANELA_CONTEXTO'])


## 7. Q5 — Movimentações

Q5 volta à maior fonte do projeto. A leitura usa simultaneamente:

- envelope temporal global calculado a partir de `radar_contexto`;
- filtros funcionais da Q5;
- particionamento JDBC por `NR_PTC` físico 1..100.

A primeira célula materializa a massa recebida do DB2 para medir a leitura real. A segunda aplica a janela individual ampliada em cinco dias.


In [ ]:
%%time
%%spark
# Q5.1 — leitura física do envelope global por NR_PTC.
t0_etapa = time.perf_counter()

e = spark.sql("""
SELECT
    DATE_SUB(MIN(DT_REF_INI),5) AS DT_MIN,
    DATE_ADD(MAX(DT_REF_FIM),5) AS DT_MAX
FROM radar_contexto
WHERE DT_REF_INI IS NOT NULL
""").first()

if e['DT_MIN'] is None:
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW radar_q5_raw AS
    SELECT
        CAST(NULL AS SMALLINT) NR_PTC,
        CAST(NULL AS BIGINT) NR_TRAN_INST_PCT,
        CAST(NULL AS INT) CD_CLI,
        CAST(NULL AS DATE) DT_TRAN,
        CAST(NULL AS STRING) CD_NTZ_CTB_TRAN,
        CAST(NULL AS INT) CD_CTGR_TRAN_OGNL,
        CAST(NULL AS STRING) CD_TIP_MOE_CRR,
        CAST(NULL AS DECIMAL(25,2)) VL_TRAN,
        CAST(NULL AS BIGINT) NR_MCA_PCT_OPB
    WHERE 1=0
    """)
else:
    df_q5 = conector_db2.sql(f"""
    SELECT
        NR_PTC,
        NR_TRAN_INST_PCT,
        CD_CLI,
        DT_TRAN,
        CD_NTZ_CTB_TRAN,
        CD_CTGR_TRAN_OGNL,
        CD_TIP_MOE_CRR,
        VL_TRAN,
        NR_MCA_PCT_OPB
    FROM DB2GFP.TRAN_RLZD_INST_PCT
    WHERE CD_CLI IS NOT NULL
      AND CD_EST_TRAN_INST = 0
      AND DT_TRAN >= DATE('{e['DT_MIN']}')
      AND DT_TRAN <= DATE('{e['DT_MAX']}')
      AND (
            CD_NTZ_CTB_TRAN='C'
            OR (CD_NTZ_CTB_TRAN='D' AND IN_VSLO_CSM='S')
          )
    """,
        fetchsize=FETCHSIZE,
        query_timeout=QUERY_TIMEOUT,
        partition_column='NR_PTC',
        lower_bound=NR_PTC_LOWER_BOUND,
        upper_bound=NR_PTC_UPPER_BOUND,
        num_partitions=Q5_JDBC_PARTITIONS,
    )
    df_q5.createOrReplaceTempView('radar_q5_raw')

spark.sql('CACHE TABLE radar_q5_raw')
q5_raw_n = spark.sql('SELECT COUNT(*) N FROM radar_q5_raw').first()['N']
metricas_execucao['Q5_LEITURA'] = {
    'segundos': round(time.perf_counter()-t0_etapa,3),
    'registros': int(q5_raw_n),
    'dt_min': str(e['DT_MIN']) if e['DT_MIN'] is not None else None,
    'dt_max': str(e['DT_MAX']) if e['DT_MAX'] is not None else None,
}
print('[Q5 LEITURA]', metricas_execucao['Q5_LEITURA'])


In [ ]:
%%time
%%spark
# Q5.2 — Aplicar a janela individual ampliada em cinco dias.
t0_etapa = time.perf_counter()
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_movimentos AS
SELECT q.NR_TRAN_INST_PCT, CAST(q.CD_CLI AS INT) CD_CLI, CAST(q.DT_TRAN AS DATE) DT_TRAN,
       q.CD_NTZ_CTB_TRAN, q.CD_CTGR_TRAN_OGNL, q.CD_TIP_MOE_CRR, q.VL_TRAN, q.NR_MCA_PCT_OPB,
       CASE WHEN q.DT_TRAN BETWEEN c.DT_REF_INI AND c.DT_REF_FIM THEN 'S' ELSE 'N' END IN_JANELA
FROM radar_q5_raw q
INNER JOIN radar_contexto c ON q.CD_CLI=c.CD_CLI
WHERE c.DT_REF_INI IS NOT NULL
  AND q.DT_TRAN BETWEEN DATE_SUB(c.DT_REF_INI,5) AND DATE_ADD(c.DT_REF_FIM,5)
""")
# Congelar movimentos para que reconciliação nunca reabra a leitura JDBC da Q5.
df_movimentos_snapshot = spark.table('radar_movimentos').localCheckpoint(eager=True)
df_movimentos_snapshot.createOrReplaceTempView('radar_movimentos')

g=spark.sql("SELECT COUNT(*) N, COUNT(DISTINCT named_struct('c',CD_CLI,'i',NR_TRAN_INST_PCT)) U, SUM(CASE WHEN NR_TRAN_INST_PCT IS NULL THEN 1 ELSE 0 END) Z FROM radar_movimentos").first()
if g['N']!=g['U'] or g['Z']!=0: raise RuntimeError(f'Identidade de movimento inválida: {g}')
spark.sql('UNCACHE TABLE radar_q5_raw')
print('Movimentos congelados:', g['N'])
metricas_execucao['Q5_JANELA'] = {'segundos': round(time.perf_counter()-t0_etapa,3), 'movimentos': int(g['N'])}
print('[Q5 JANELA]', metricas_execucao['Q5_JANELA'])


## 8. Reconciliação — exata, residual, borda e efetivos


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
# Reconciliação exata — Parear créditos e débitos do mesmo grupo.
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_pares_exatos AS
WITH grupos AS (
    SELECT
        CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA,
        sort_array(filter(collect_list(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN
            named_struct('id', CAST(NR_TRAN_INST_PCT AS BIGINT), 'bank', CAST(NR_MCA_PCT_OPB AS STRING)) END), x -> x IS NOT NULL)) AS C,
        sort_array(filter(collect_list(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN
            named_struct('id', CAST(NR_TRAN_INST_PCT AS BIGINT), 'bank', CAST(NR_MCA_PCT_OPB AS STRING)) END), x -> x IS NOT NULL)) AS D
    FROM radar_movimentos
    WHERE NR_TRAN_INST_PCT IS NOT NULL
      AND DT_TRAN IS NOT NULL
      AND VL_TRAN IS NOT NULL
      AND CD_TIP_MOE_CRR IS NOT NULL
      AND CD_NTZ_CTB_TRAN IN ('C','D')
    GROUP BY CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA
), candidatos AS (
    SELECT * FROM grupos WHERE size(C) > 0 AND size(D) > 0
), matching AS (
    SELECT
        CD_CLI, IN_JANELA, C, D,
        aggregate(
            sequence(0, size(C)-1),
            named_struct('mc', array_repeat(-1, size(C)), 'md', array_repeat(-1, size(D))),
            (m, ci0) ->
                element_at(
                    transform(
                        array(
                            aggregate(
                                sequence(1, size(C)),
                                named_struct(
                                    'queue', array(ci0),
                                    'qpos', 0,
                                    'seen_c', array(ci0),
                                    'seen_d', CAST(array() AS ARRAY<INT>),
                                    'parent_d', array_repeat(-1, size(D)),
                                    'free_d', -1,
                                    'mc', m.mc,
                                    'md', m.md
                                ),
                                (b, passo) ->
                                    CASE
                                      WHEN b.free_d <> -1 OR b.qpos >= size(b.queue) THEN b
                                      ELSE element_at(
                                        transform(
                                          array(
                                            aggregate(
                                              sequence(0, size(D)-1),
                                              named_struct(
                                                'queue', b.queue,
                                                'qpos', b.qpos + 1,
                                                'seen_c', b.seen_c,
                                                'seen_d', b.seen_d,
                                                'parent_d', b.parent_d,
                                                'free_d', b.free_d,
                                                'mc', b.mc,
                                                'md', b.md,
                                                'current_ci', b.queue[b.qpos]
                                              ),
                                              (s, di) ->
                                                CASE
                                                  WHEN s.free_d <> -1
                                                    OR array_contains(s.seen_d, di)
                                                    OR C[s.current_ci].bank IS NULL
                                                    OR D[di].bank IS NULL
                                                    OR C[s.current_ci].bank = D[di].bank
                                                  THEN s
                                                  WHEN s.md[di] = -1 THEN named_struct(
                                                    'queue', s.queue,
                                                    'qpos', s.qpos,
                                                    'seen_c', s.seen_c,
                                                    'seen_d', concat(s.seen_d, array(di)),
                                                    'parent_d', transform(s.parent_d, (x,k) -> IF(k=di, s.current_ci, x)),
                                                    'free_d', di,
                                                    'mc', s.mc,
                                                    'md', s.md,
                                                    'current_ci', s.current_ci
                                                  )
                                                  ELSE named_struct(
                                                    'queue', IF(array_contains(s.seen_c, s.md[di]), s.queue, concat(s.queue, array(s.md[di]))),
                                                    'qpos', s.qpos,
                                                    'seen_c', IF(array_contains(s.seen_c, s.md[di]), s.seen_c, concat(s.seen_c, array(s.md[di]))),
                                                    'seen_d', concat(s.seen_d, array(di)),
                                                    'parent_d', transform(s.parent_d, (x,k) -> IF(k=di, s.current_ci, x)),
                                                    'free_d', s.free_d,
                                                    'mc', s.mc,
                                                    'md', s.md,
                                                    'current_ci', s.current_ci
                                                  )
                                                END
                                            )
                                          ),
                                          z -> named_struct(
                                            'queue', z.queue, 'qpos', z.qpos, 'seen_c', z.seen_c,
                                            'seen_d', z.seen_d, 'parent_d', z.parent_d,
                                            'free_d', z.free_d, 'mc', z.mc, 'md', z.md
                                          )
                                        ), 1)
                                    END,
                                b -> b
                            )
                        ),
                        bfs -> element_at(
                            transform(
                                array(
                                    aggregate(
                                        sequence(1, size(C)+size(D)),
                                        named_struct(
                                            'mc', bfs.mc,
                                            'md', bfs.md,
                                            'di', bfs.free_d,
                                            'done', bfs.free_d = -1,
                                            'parent_d', bfs.parent_d
                                        ),
                                        (a, passo2) ->
                                            CASE WHEN a.done THEN a ELSE
                                              element_at(
                                                transform(
                                                  array(a.parent_d[a.di]),
                                                  ci -> element_at(
                                                    transform(
                                                      array(a.mc[ci]),
                                                      prev_di -> named_struct(
                                                        'mc', transform(a.mc, (x,k) -> IF(k=ci, a.di, x)),
                                                        'md', transform(a.md, (x,k) -> IF(k=a.di, ci, x)),
                                                        'di', prev_di,
                                                        'done', prev_di = -1,
                                                        'parent_d', a.parent_d
                                                      )
                                                    ), 1)
                                                ), 1)
                                            END,
                                        a -> a
                                    )
                                ),
                                aug -> named_struct('mc', aug.mc, 'md', aug.md)
                            ), 1)
                    ), 1)
        ) AS M
    FROM candidatos
)
SELECT
    CD_CLI,
    IN_JANELA,
    C[ci].id AS ID_CREDITO,
    D[M.mc[ci]].id AS ID_DEBITO
FROM matching
LATERAL VIEW explode(sequence(0, size(M.mc)-1)) e AS ci
WHERE M.mc[ci] >= 0
""")
spark.sql('CACHE TABLE radar_pares_exatos')
print('Pares exatos:', spark.sql('SELECT COUNT(*) N FROM radar_pares_exatos').first()['N'])
metricas_execucao['RECON_EXATA'] = {'segundos': round(time.perf_counter() - t0_etapa, 3)}
print('[TEMPO RECON_EXATA]', metricas_execucao['RECON_EXATA']['segundos'], 's')


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
# Residual — Remover os IDs consumidos nos pares exatos.
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_ids_exatos AS
SELECT CD_CLI, ID_CREDITO NR_TRAN_INST_PCT FROM radar_pares_exatos
UNION ALL
SELECT CD_CLI, ID_DEBITO NR_TRAN_INST_PCT FROM radar_pares_exatos
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_residual AS
SELECT m.*
FROM radar_movimentos m
LEFT ANTI JOIN radar_ids_exatos x
  ON m.CD_CLI=x.CD_CLI AND m.NR_TRAN_INST_PCT=x.NR_TRAN_INST_PCT
""")
spark.sql('CACHE TABLE radar_residual')
print('Residual:', spark.sql('SELECT COUNT(*) N FROM radar_residual').first()['N'])
metricas_execucao['RECON_RESIDUAL'] = {'segundos': round(time.perf_counter() - t0_etapa, 3)}
print('[TEMPO RECON_RESIDUAL]', metricas_execucao['RECON_RESIDUAL']['segundos'], 's')


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
# Reconciliação de borda — Parear movimentos internos e externos residuais.
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_pares_borda AS
WITH dentro AS (
    SELECT
        CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CD_NTZ_CTB_TRAN,
        sort_array(collect_list(named_struct(
            'dt', DT_TRAN,
            'id', CAST(NR_TRAN_INST_PCT AS BIGINT),
            'bank', CAST(NR_MCA_PCT_OPB AS STRING)
        ))) AS DENTRO
    FROM radar_residual
    WHERE IN_JANELA = 'S'
      AND DT_TRAN IS NOT NULL
      AND VL_TRAN IS NOT NULL
      AND CD_TIP_MOE_CRR IS NOT NULL
      AND CD_NTZ_CTB_TRAN IN ('C','D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CD_NTZ_CTB_TRAN
), fora AS (
    SELECT
        CD_CLI, VL_TRAN, CD_TIP_MOE_CRR,
        CASE WHEN CD_NTZ_CTB_TRAN='C' THEN 'D' ELSE 'C' END AS NTZ_DENTRO,
        sort_array(collect_list(named_struct(
            'dt', DT_TRAN,
            'id', CAST(NR_TRAN_INST_PCT AS BIGINT),
            'bank', CAST(NR_MCA_PCT_OPB AS STRING)
        ))) AS FORA
    FROM radar_residual
    WHERE IN_JANELA = 'N'
      AND DT_TRAN IS NOT NULL
      AND VL_TRAN IS NOT NULL
      AND CD_TIP_MOE_CRR IS NOT NULL
      AND CD_NTZ_CTB_TRAN IN ('C','D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR,
             CASE WHEN CD_NTZ_CTB_TRAN='C' THEN 'D' ELSE 'C' END
), grupos AS (
    SELECT d.CD_CLI, d.VL_TRAN, d.CD_TIP_MOE_CRR, d.CD_NTZ_CTB_TRAN, d.DENTRO, f.FORA
    FROM dentro d
    INNER JOIN fora f
      ON d.CD_CLI=f.CD_CLI
     AND d.VL_TRAN=f.VL_TRAN
     AND d.CD_TIP_MOE_CRR=f.CD_TIP_MOE_CRR
     AND d.CD_NTZ_CTB_TRAN=f.NTZ_DENTRO
), dp AS (
    SELECT
        *,
        aggregate(
            sequence(size(DENTRO)*size(FORA)-1, 0, -1),
            array_repeat(
                named_struct(
                    'n', 0,
                    'cost', 0,
                    'pairs', CAST(array() AS ARRAY<STRUCT<NR_TRAN_DENTRO:BIGINT,NR_TRAN_FORA:BIGINT,DT_TRAN_DENTRO:DATE,DT_TRAN_FORA:DATE,DIF_DIAS:INT>>)
                ),
                (size(DENTRO)+1)*(size(FORA)+1)
            ),
            (estado, k) -> element_at(
                transform(
                    array(named_struct(
                        'i', CAST(floor(k / size(FORA)) AS INT),
                        'j', CAST(pmod(k, size(FORA)) AS INT),
                        'w', size(FORA)+1
                    )),
                    ij -> element_at(
                        transform(
                            array(
                                estado[(ij.i+1)*ij.w + ij.j],
                                estado[ij.i*ij.w + (ij.j+1)],
                                estado[(ij.i+1)*ij.w + (ij.j+1)]
                            ),
                            refs -> element_at(
                                transform(
                                    array_sort(filter(array(
                                        named_struct(
                                            'neg_n', -refs[0].n,
                                            'cost', refs[0].cost,
                                            'key', transform(refs[0].pairs, p -> named_struct('a',p.NR_TRAN_DENTRO,'b',p.NR_TRAN_FORA)),
                                            'sol', refs[0]
                                        ),
                                        named_struct(
                                            'neg_n', -refs[1].n,
                                            'cost', refs[1].cost,
                                            'key', transform(refs[1].pairs, p -> named_struct('a',p.NR_TRAN_DENTRO,'b',p.NR_TRAN_FORA)),
                                            'sol', refs[1]
                                        ),
                                        CASE WHEN
                                            abs(datediff(DENTRO[ij.i].dt, FORA[ij.j].dt)) BETWEEN 1 AND 5
                                            AND DENTRO[ij.i].bank IS NOT NULL
                                            AND FORA[ij.j].bank IS NOT NULL
                                            AND DENTRO[ij.i].bank <> FORA[ij.j].bank
                                        THEN named_struct(
                                            'neg_n', -(refs[2].n + 1),
                                            'cost', refs[2].cost + abs(datediff(DENTRO[ij.i].dt, FORA[ij.j].dt)),
                                            'key', transform(
                                                concat(
                                                    array(named_struct(
                                                        'NR_TRAN_DENTRO', DENTRO[ij.i].id,
                                                        'NR_TRAN_FORA', FORA[ij.j].id,
                                                        'DT_TRAN_DENTRO', DENTRO[ij.i].dt,
                                                        'DT_TRAN_FORA', FORA[ij.j].dt,
                                                        'DIF_DIAS', CAST(abs(datediff(DENTRO[ij.i].dt, FORA[ij.j].dt)) AS INT)
                                                    )),
                                                    refs[2].pairs
                                                ),
                                                p -> named_struct('a',p.NR_TRAN_DENTRO,'b',p.NR_TRAN_FORA)
                                            ),
                                            'sol', named_struct(
                                                'n', refs[2].n + 1,
                                                'cost', refs[2].cost + abs(datediff(DENTRO[ij.i].dt, FORA[ij.j].dt)),
                                                'pairs', concat(
                                                    array(named_struct(
                                                        'NR_TRAN_DENTRO', DENTRO[ij.i].id,
                                                        'NR_TRAN_FORA', FORA[ij.j].id,
                                                        'DT_TRAN_DENTRO', DENTRO[ij.i].dt,
                                                        'DT_TRAN_FORA', FORA[ij.j].dt,
                                                        'DIF_DIAS', CAST(abs(datediff(DENTRO[ij.i].dt, FORA[ij.j].dt)) AS INT)
                                                    )),
                                                    refs[2].pairs
                                                )
                                            )
                                        ) END
                                    ), x -> x IS NOT NULL)),
                                    melhor -> transform(
                                        estado,
                                        (celula,pos) -> IF(pos = ij.i*ij.w + ij.j, melhor[0].sol, celula)
                                    )
                                ), 1)
                            )
                        , 1)
                ), 1),
            estado -> estado[0].pairs
        ) AS PARES
    FROM grupos
)
SELECT CD_CLI, p.*
FROM dp
LATERAL VIEW explode(PARES) e AS p
""")
spark.sql('CACHE TABLE radar_pares_borda')
print('Pares de borda:', spark.sql('SELECT COUNT(*) N FROM radar_pares_borda').first()['N'])
metricas_execucao['RECON_BORDA'] = {'segundos': round(time.perf_counter() - t0_etapa, 3)}
print('[TEMPO RECON_BORDA]', metricas_execucao['RECON_BORDA']['segundos'], 's')


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
# Efetivos — Manter movimentos da janela não consumidos.
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_ids_borda AS
SELECT CD_CLI, NR_TRAN_DENTRO NR_TRAN_INST_PCT FROM radar_pares_borda
UNION ALL
SELECT CD_CLI, NR_TRAN_FORA NR_TRAN_INST_PCT FROM radar_pares_borda
""")
reuso=spark.sql("""
SELECT COUNT(*) N FROM (
  SELECT CD_CLI,NR_TRAN_INST_PCT,COUNT(*) Q FROM (
    SELECT * FROM radar_ids_exatos UNION ALL SELECT * FROM radar_ids_borda
  ) x GROUP BY CD_CLI,NR_TRAN_INST_PCT HAVING COUNT(*)>1
) z
""").first()['N']
if reuso: raise RuntimeError(f'Reconciliação reutilizou {reuso} transação(ões).')
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_efetivos AS
SELECT r.*
FROM radar_residual r
LEFT ANTI JOIN radar_ids_borda b
  ON r.CD_CLI=b.CD_CLI AND r.NR_TRAN_INST_PCT=b.NR_TRAN_INST_PCT
WHERE r.IN_JANELA='S'
""")
df_efetivos_snapshot = spark.table('radar_efetivos').localCheckpoint(eager=True)
df_efetivos_snapshot.createOrReplaceTempView('radar_efetivos')
print('Efetivos congelados:', spark.sql('SELECT COUNT(*) N FROM radar_efetivos').first()['N'])
spark.sql('UNCACHE TABLE radar_movimentos')
spark.sql('UNCACHE TABLE radar_residual')
spark.sql('UNCACHE TABLE radar_pares_exatos')
spark.sql('UNCACHE TABLE radar_pares_borda')
metricas_execucao['RECON_EFETIVOS'] = {'segundos': round(time.perf_counter() - t0_etapa, 3)}
print('[TEMPO RECON_EFETIVOS]', metricas_execucao['RECON_EFETIVOS']['segundos'], 's')


## 9. Classificação, agregação, motor e resultado final


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
# Classificação — Aplicar o mapa Radar aos movimentos efetivos.
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_categorias AS
SELECT *
FROM VALUES
    (NULL, 0, 'Sem categoria', 0, 'Sem categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 1, 'Salário', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 2, 'Vale Alimentação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 3, 'Restituição de IR', 0, 'Não pertence', 2, 'Estorno', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 4, 'Bonificação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 5, 'Outros Rendimentos', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 6, 'Água', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 7, 'Eletricidade e Gás', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 9, 'Compra de Imóvel', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 10, 'Aluguel e Condomínio', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 11, 'Móveis e Utensílios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 12, 'Serviços e Manutenção', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 13, 'Empregados', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 14, 'Animais e Pets', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 15, 'Educação Superior', 1, 'Pagamentos efetuados', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 16, 'Colégio', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 17, 'Idiomas', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 18, 'Publicações e Papelaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 20, 'Outros Gastos, Educação', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 21, 'Viagens e Lazer', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 22, 'Esportes e Academia', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 25, 'Cultura e Entretenimento', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 27, 'Plano de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 28, 'Serviços de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 29, 'Dentista', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 30, 'Farmácias e Drogarias', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 32, 'Feira e Supermercado', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 35, 'Bar, Rest. e Padaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 36, 'Compra de Veículo', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 37, 'Combustível', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 38, 'Estacionamento e Pedágio', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 39, 'Seguro de Veículo', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 40, 'Serviços e Manutenção', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 41, 'Transporte Urbano e Apps', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 42, 'Vestuário e Acessórios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 43, 'Cuidado Pessoal e Beleza', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 44, 'Compras Diversas', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 45, 'Pensão Alimentícia', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 46, 'Seguros e Previdência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 47, 'Doação', 4, 'Doações efetuadas', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 48, 'Gasto com Familiares', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 49, 'Presentes', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 51, 'Telefonia e Internet', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 53, 'Assinatura TV e Streaming', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 54, 'IPTU', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 55, 'IPVA e Gastos Detran', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 56, 'Imposto de Renda', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 57, 'ISS(Imposto sobre Serviços)', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 58, 'GPS(Guia de Previdência Social)', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 59, 'Serviços Financeiros', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 60, 'Serviços Diversos', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 61, 'Jogos e Loterias', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    (NULL, 0, 'Sem categoria', 83, 'Sem Categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('D', 12, 'Fatura', 111, 'Cartão de Crédito', 0, 'Não pertence', 9, 'Obrigações', 'N', 'N', 'N'),
    ('D', 11, 'Outros', 279, 'Gastos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('C', 14, 'Agro', 300, 'Receitas Agro', 0, 'Não pertence', 1, 'Renda', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 310, 'Criações', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 330, 'Cultivos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 350, 'Insumos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 370, 'Apoio Produtivo', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 10, 'Tarifas e impostos', 3787, 'IOF', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 3788, 'Encargos e Tarifas', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 3790, 'Seguro Residencial', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 4417, 'Empréstimos e Prestações', 3, 'Dívidas e ônus reais', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39434, 'Cheque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39435, 'Saque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39436, 'Transferência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39437, 'Boletos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 13, 'Investimentos', 448977, 'Aplicação', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'N'),
    ('C', 13, 'Investimentos', 448978, 'Resgate de Investimentos', 0, 'Não pertence', 3, 'Resgate', 'N', 'S', 'N')
AS c(
    TIPO, CD_GRUPO, TX_GRUPO, CD_CATEGORIA, TX_CATEGORIA,
    CD_IR, TX_IR, CD_CLASS_RADAR, TX_CLASS_RADAR,
    IN_AGRO, IN_PARTICIPA_CALCULO, IN_PARTICIPA_ORCAMENTO
)
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_classificados AS
SELECT e.*,
       CASE WHEN c.CD_CATEGORIA IS NOT NULL THEN TRUE ELSE FALSE END TEM_CATEGORIA,
       COALESCE(c.CD_CLASS_RADAR,0) CD_CLASS_RADAR,
       COALESCE(c.IN_AGRO,'N') IN_AGRO,
       COALESCE(c.IN_PARTICIPA_CALCULO,'N') IN_PARTICIPA_CALCULO,
       COALESCE(c.IN_PARTICIPA_ORCAMENTO,'N') IN_PARTICIPA_ORCAMENTO
FROM radar_efetivos e
LEFT JOIN radar_categorias c
  ON e.CD_CTGR_TRAN_OGNL=c.CD_CATEGORIA AND e.CD_NTZ_CTB_TRAN=c.TIPO
""")
spark.sql('CACHE TABLE radar_classificados')
print('Classificados:', spark.sql('SELECT COUNT(*) N FROM radar_classificados').first()['N'])
spark.sql('UNCACHE TABLE radar_efetivos')
metricas_execucao['CLASSIFICACAO'] = {'segundos': round(time.perf_counter() - t0_etapa, 3)}
print('[TEMPO CLASSIFICACAO]', metricas_execucao['CLASSIFICACAO']['segundos'], 's')


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
# Agregação — Produzir uma linha financeira por cliente.
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_agregado AS
SELECT CD_CLI,
    CASE WHEN COUNT(*)=0 THEN NULL WHEN COUNT(DISTINCT CD_TIP_MOE_CRR)=1 AND MAX(CD_TIP_MOE_CRR)='BRL' THEN 'S' ELSE 'N' END FL_SOMENTE_BRL,
    CASE WHEN COUNT(CASE WHEN CD_TIP_MOE_CRR='BRL' THEN 1 END)=0 THEN NULL
         WHEN COUNT(CASE WHEN CD_TIP_MOE_CRR='BRL' AND IN_AGRO='S' THEN 1 END)>0 THEN 'S' ELSE 'N' END FL_TEM_MOV_AGRO,
    CAST(COUNT(CASE WHEN CD_TIP_MOE_CRR='BRL' THEN 1 END) AS BIGINT) QT_TRANS_TOTAL,
    CAST(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='C' THEN 1 WHEN CD_TIP_MOE_CRR='BRL' THEN 0 END) AS BIGINT) QT_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='D' THEN 1 WHEN CD_TIP_MOE_CRR='BRL' THEN 0 END) AS BIGINT) QT_TRANS_SAI,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='C' AND CD_CLASS_RADAR=1 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_ENT_REN,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='C' AND CD_CLASS_RADAR=2 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_ENT_EST,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='C' AND CD_CLASS_RADAR=3 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_ENT_RESG,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='C' AND CD_CLASS_RADAR=0 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_ENT_OUT,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='C' AND CD_CLASS_RADAR=4 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_ENT_CRED,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='D' AND CD_CLASS_RADAR=5 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_SAI_IND,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='D' AND CD_CLASS_RADAR=6 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_SAI_ESS,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='D' AND CD_CLASS_RADAR=7 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_SAI_NAO_ESS,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='D' AND CD_CLASS_RADAR=8 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_SAI_FUT,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='D' AND CD_CLASS_RADAR=9 AND IN_PARTICIPA_CALCULO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_SAI_OBR,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='C' AND IN_PARTICIPA_ORCAMENTO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_ENT_TOTAL,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='D' AND IN_PARTICIPA_ORCAMENTO='S' THEN VL_TRAN END),0) AS DECIMAL(25,2)) VL_SAI_TOTAL,
    CAST(COALESCE(SUM(CASE WHEN CD_TIP_MOE_CRR='BRL' AND CD_NTZ_CTB_TRAN='C' AND TEM_CATEGORIA THEN VL_TRAN END),0) AS DECIMAL(25,2)) ENTRADAS_REALIZADAS
FROM radar_classificados
GROUP BY CD_CLI
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_base AS
SELECT c.*,
       a.FL_SOMENTE_BRL, a.FL_TEM_MOV_AGRO,
       COALESCE(a.QT_TRANS_TOTAL,CAST(0 AS BIGINT)) QT_TRANS_TOTAL,
       a.QT_TRANS_ENT, a.QT_TRANS_SAI,
       CAST(COALESCE(a.VL_ENT_TOTAL,0) AS DECIMAL(25,2)) VL_TRANS_ENT,
       CAST(COALESCE(a.VL_SAI_TOTAL,0) AS DECIMAL(25,2)) VL_TRANS_SAI,
       CAST(COALESCE(a.VL_ENT_REN,0) AS DECIMAL(25,2)) VL_ENT_REN,
       CAST(COALESCE(a.VL_ENT_EST,0) AS DECIMAL(25,2)) VL_ENT_EST,
       CAST(COALESCE(a.VL_ENT_RESG,0) AS DECIMAL(25,2)) VL_ENT_RESG,
       CAST(COALESCE(a.VL_ENT_OUT,0) AS DECIMAL(25,2)) VL_ENT_OUT,
       CAST(COALESCE(a.VL_ENT_CRED,0) AS DECIMAL(25,2)) VL_ENT_CRED,
       CAST(COALESCE(a.VL_ENT_TOTAL,0) AS DECIMAL(25,2)) VL_ENT_TOTAL,
       CAST(COALESCE(a.VL_SAI_IND,0) AS DECIMAL(25,2)) VL_SAI_IND,
       CAST(COALESCE(a.VL_SAI_ESS,0) AS DECIMAL(25,2)) VL_SAI_ESS,
       CAST(COALESCE(a.VL_SAI_NAO_ESS,0) AS DECIMAL(25,2)) VL_SAI_NAO_ESS,
       CAST(COALESCE(a.VL_SAI_FUT,0) AS DECIMAL(25,2)) VL_SAI_FUT,
       CAST(COALESCE(a.VL_SAI_OBR,0) AS DECIMAL(25,2)) VL_SAI_OBR,
       CAST(COALESCE(a.VL_SAI_TOTAL,0) AS DECIMAL(25,2)) VL_SAI_TOTAL,
       CAST(COALESCE(a.ENTRADAS_REALIZADAS,0) AS DECIMAL(25,2)) ENTRADAS_REALIZADAS
FROM radar_contexto c
LEFT JOIN radar_agregado a ON c.CD_CLI=a.CD_CLI
""")
spark.sql('CACHE TABLE radar_base')
g=spark.sql('SELECT COUNT(*) N, COUNT(DISTINCT CD_CLI) U FROM radar_base').first()
if g['N']!=g['U']: raise RuntimeError(f'Agregação multiplicou clientes: {g}')
spark.sql('UNCACHE TABLE radar_classificados')
print('Agregação:', g['N'], 'clientes')
metricas_execucao['AGREGACAO'] = {'segundos': round(time.perf_counter() - t0_etapa, 3)}
print('[TEMPO AGREGACAO]', metricas_execucao['AGREGACAO']['segundos'], 's')


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
# Motor — Calcular oficial e os dois cenários com a mesma sequência SQL.
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_motor_entrada AS
SELECT b.CD_CLI, b.QT_TRANS_TOTAL, b.CD_MAC_PRFL_CLI,
       b.VL_SAI_TOTAL, b.VL_SAI_IND, b.VL_SAI_ESS, b.VL_SAI_NAO_ESS, b.VL_SAI_FUT, b.VL_SAI_OBR,
       s.CENARIO, s.BASE_ORCAMENTO, s.BASE_PERCENTUAIS
FROM radar_base b
LATERAL VIEW STACK(3,
    'OFICIAL', CAST(b.VL_ENT_TOTAL AS DECIMAL(25,2)), CAST(b.VL_REN_PRES AS DECIMAL(25,2)),
    'RENDA_PRESUMIDA', CASE WHEN b.DT_REF_INI IS NOT NULL AND b.DT_REF_FIM IS NOT NULL THEN CAST(b.VL_REN_PRES AS DECIMAL(25,2)) END, CASE WHEN b.DT_REF_INI IS NOT NULL AND b.DT_REF_FIM IS NOT NULL THEN CAST(b.VL_REN_PRES AS DECIMAL(25,2)) END,
    'ENTRADAS_REALIZADAS', CASE WHEN b.DT_REF_INI IS NOT NULL AND b.DT_REF_FIM IS NOT NULL THEN CAST(b.ENTRADAS_REALIZADAS AS DECIMAL(25,2)) END, CASE WHEN b.DT_REF_INI IS NOT NULL AND b.DT_REF_FIM IS NOT NULL THEN CAST(b.ENTRADAS_REALIZADAS AS DECIMAL(25,2)) END
) s AS CENARIO, BASE_ORCAMENTO, BASE_PERCENTUAIS
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_motor_percentuais AS
SELECT
    CD_CLI, CENARIO, QT_TRANS_TOTAL, CD_MAC_PRFL_CLI,
    VL_SAI_TOTAL, VL_SAI_IND, VL_SAI_ESS, VL_SAI_NAO_ESS, VL_SAI_FUT, VL_SAI_OBR,
    BASE_ORCAMENTO, BASE_PERCENTUAIS,
    CASE WHEN BASE_ORCAMENTO IS NULL OR VL_SAI_TOTAL IS NULL THEN NULL
         ELSE CAST(ROUND(BASE_ORCAMENTO - VL_SAI_TOTAL, 2) AS DECIMAL(25,2)) END AS VL_RES_ORC,
    CASE WHEN (QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0) OR (BASE_ORCAMENTO) IS NULL OR (BASE_ORCAMENTO) = 0 THEN NULL WHEN ABS(ROUND(CAST((VL_SAI_TOTAL) AS DECIMAL(38,12)) / CAST((BASE_ORCAMENTO) AS DECIMAL(38,12)), 6)) >= 1000 THEN NULL ELSE CAST(ROUND(CAST((VL_SAI_TOTAL) AS DECIMAL(38,12)) / CAST((BASE_ORCAMENTO) AS DECIMAL(38,12)), 6) AS DECIMAL(9,6)) END AS PC_SAI_ENT,
    CASE WHEN (BASE_PERCENTUAIS <= 0) OR (BASE_PERCENTUAIS) IS NULL OR (BASE_PERCENTUAIS) = 0 THEN NULL WHEN ABS(ROUND(CAST((VL_SAI_IND) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6)) >= 1000 THEN NULL ELSE CAST(ROUND(CAST((VL_SAI_IND) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6) AS DECIMAL(9,6)) END AS PC_SAI_IND,
    CASE WHEN (BASE_PERCENTUAIS <= 0) OR (BASE_PERCENTUAIS) IS NULL OR (BASE_PERCENTUAIS) = 0 THEN NULL WHEN ABS(ROUND(CAST((VL_SAI_ESS) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6)) >= 1000 THEN NULL ELSE CAST(ROUND(CAST((VL_SAI_ESS) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6) AS DECIMAL(9,6)) END AS PC_SAI_ESS,
    CASE WHEN (BASE_PERCENTUAIS <= 0) OR (BASE_PERCENTUAIS) IS NULL OR (BASE_PERCENTUAIS) = 0 THEN NULL WHEN ABS(ROUND(CAST((VL_SAI_NAO_ESS) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6)) >= 1000 THEN NULL ELSE CAST(ROUND(CAST((VL_SAI_NAO_ESS) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6) AS DECIMAL(9,6)) END AS PC_SAI_NAO_ESS,
    CASE WHEN (BASE_PERCENTUAIS <= 0) OR (BASE_PERCENTUAIS) IS NULL OR (BASE_PERCENTUAIS) = 0 THEN NULL WHEN ABS(ROUND(CAST((VL_SAI_FUT) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6)) >= 1000 THEN NULL ELSE CAST(ROUND(CAST((VL_SAI_FUT) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6) AS DECIMAL(9,6)) END AS PC_SAI_FUT,
    CASE WHEN (BASE_PERCENTUAIS <= 0) OR (BASE_PERCENTUAIS) IS NULL OR (BASE_PERCENTUAIS) = 0 THEN NULL WHEN ABS(ROUND(CAST((VL_SAI_OBR) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6)) >= 1000 THEN NULL ELSE CAST(ROUND(CAST((VL_SAI_OBR) AS DECIMAL(38,12)) / CAST((BASE_PERCENTUAIS) AS DECIMAL(38,12)), 6) AS DECIMAL(9,6)) END AS PC_SAI_OBR
FROM radar_motor_entrada
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_motor_faixa AS
SELECT
    *,
    CASE
      WHEN PC_SAI_ENT IS NULL THEN NULL
      WHEN PC_SAI_ENT BETWEEN CAST(0.950000 AS DECIMAL(9,6)) AND CAST(1.050000 AS DECIMAL(9,6)) THEN 0
      WHEN PC_SAI_ENT > CAST(1.050000 AS DECIMAL(9,6)) AND PC_SAI_ENT <= CAST(1.250000 AS DECIMAL(9,6)) THEN 1
      WHEN PC_SAI_ENT > CAST(1.250000 AS DECIMAL(9,6)) THEN 2
      WHEN PC_SAI_ENT >= CAST(0.750000 AS DECIMAL(9,6)) AND PC_SAI_ENT < CAST(0.950000 AS DECIMAL(9,6)) THEN 3
      ELSE 4
    END AS CD_FAIXA_ORC
FROM radar_motor_percentuais
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_motor_pontos AS
SELECT
    *,
    CASE WHEN CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC=0 THEN 0 WHEN CD_FAIXA_ORC IN (1,2) THEN 2 ELSE 1 END AS CD_RES_ORC,
    CASE WHEN CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC=0 THEN 'Neutro' WHEN CD_FAIXA_ORC IN (1,2) THEN 'Deficitário' ELSE 'Superavitário' END AS TX_RES_ORC,
    CASE WHEN CD_FAIXA_ORC IN (1,3) THEN 'Moderado' WHEN CD_FAIXA_ORC IN (2,4) THEN 'Acentuado' ELSE NULL END AS TX_STS_RES,
    CASE WHEN CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC=0 THEN 'Neutro' WHEN CD_FAIXA_ORC=1 THEN 'Deficitário Moderado' WHEN CD_FAIXA_ORC=2 THEN 'Deficitário Acentuado' WHEN CD_FAIXA_ORC=3 THEN 'Superavitário Moderado' ELSE 'Superavitário Acentuado' END AS TX_STS_FINAL,

    CAST(0.750000 AS DECIMAL(9,6)) AS PC_REF_IND,
    CAST(0.500000 AS DECIMAL(9,6)) AS PC_REF_ESS,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
    CAST(0.200000 AS DECIMAL(9,6)) AS PC_REF_FUT,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_OBR,

    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR BASE_PERCENTUAIS IS NULL THEN NULL
         WHEN BASE_PERCENTUAIS <= 0 THEN 0
         WHEN PC_SAI_IND IS NOT NULL AND PC_SAI_IND > CAST(0.750000 AS DECIMAL(9,6)) THEN 99 ELSE 0 END AS NR_PONT_CONC_IND,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR BASE_PERCENTUAIS IS NULL THEN NULL
         WHEN BASE_PERCENTUAIS <= 0 THEN 0
         WHEN PC_SAI_ESS IS NOT NULL AND PC_SAI_ESS < CAST(0.500000 AS DECIMAL(9,6)) THEN 0
         WHEN PC_SAI_ESS IS NOT NULL AND PC_SAI_ESS < CAST(0.750000 AS DECIMAL(9,6)) THEN 1 ELSE 2 END AS NR_PONT_CONC_ESS,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR BASE_PERCENTUAIS IS NULL THEN NULL
         WHEN BASE_PERCENTUAIS <= 0 THEN 0
         WHEN PC_SAI_NAO_ESS IS NOT NULL AND PC_SAI_NAO_ESS < CAST(0.300000 AS DECIMAL(9,6)) THEN 0
         WHEN PC_SAI_NAO_ESS IS NOT NULL AND PC_SAI_NAO_ESS < CAST(0.450000 AS DECIMAL(9,6)) THEN 1 ELSE 2 END AS NR_PONT_CONC_NAO_ESS,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR BASE_PERCENTUAIS IS NULL THEN NULL
         WHEN BASE_PERCENTUAIS <= 0 THEN 0
         WHEN PC_SAI_FUT IS NOT NULL AND PC_SAI_FUT >= CAST(0.300000 AS DECIMAL(9,6)) THEN 0
         WHEN PC_SAI_FUT IS NOT NULL AND PC_SAI_FUT >= CAST(0.200000 AS DECIMAL(9,6)) THEN 1 ELSE 2 END AS NR_PONT_CONC_FUT,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR BASE_PERCENTUAIS IS NULL THEN NULL
         WHEN BASE_PERCENTUAIS <= 0 THEN 0
         WHEN PC_SAI_OBR IS NOT NULL AND PC_SAI_OBR < CAST(0.300000 AS DECIMAL(9,6)) THEN 0
         WHEN PC_SAI_OBR IS NOT NULL AND PC_SAI_OBR < CAST(0.450000 AS DECIMAL(9,6)) THEN 1 ELSE 2 END AS NR_PONT_CONC_OBR,

    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 THEN NULL ELSE 0 END AS NR_PONT_ORC_IND,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC=2 THEN 2 WHEN CD_FAIXA_ORC IN (0,1) THEN 1 ELSE 0 END AS NR_PONT_ORC_ESS,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC=2 THEN 2 WHEN CD_FAIXA_ORC IN (0,1) THEN 1 ELSE 0 END AS NR_PONT_ORC_NAO_ESS,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC=4 THEN 2 WHEN CD_FAIXA_ORC IN (0,3) THEN 1 ELSE 0 END AS NR_PONT_ORC_FUT,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC=2 THEN 2 WHEN CD_FAIXA_ORC IN (0,1) THEN 1 ELSE 0 END AS NR_PONT_ORC_OBR,

    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 THEN NULL ELSE 0 END AS NR_PONT_PRFL_IND,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR CD_MAC_PRFL_CLI NOT IN (1,2,3) OR CD_MAC_PRFL_CLI IS NULL THEN NULL WHEN CD_MAC_PRFL_CLI=1 THEN 0 ELSE 1 END AS NR_PONT_PRFL_ESS,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR CD_MAC_PRFL_CLI NOT IN (1,2,3) OR CD_MAC_PRFL_CLI IS NULL THEN NULL WHEN CD_MAC_PRFL_CLI=1 THEN 1 ELSE 0 END AS NR_PONT_PRFL_NAO_ESS,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR CD_MAC_PRFL_CLI NOT IN (1,2,3) OR CD_MAC_PRFL_CLI IS NULL THEN NULL WHEN CD_MAC_PRFL_CLI=3 THEN 2 WHEN CD_MAC_PRFL_CLI=2 THEN 1 ELSE 0 END AS NR_PONT_PRFL_FUT,
    CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL=0 OR CD_MAC_PRFL_CLI NOT IN (1,2,3) OR CD_MAC_PRFL_CLI IS NULL THEN NULL WHEN CD_MAC_PRFL_CLI=1 THEN 2 ELSE 0 END AS NR_PONT_PRFL_OBR
FROM radar_motor_faixa
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_motor_finais AS
SELECT
    *,
    NR_PONT_CONC_IND AS NR_PONT_IND_FIM,
    CASE WHEN NR_PONT_CONC_ESS IS NULL OR NR_PONT_ORC_ESS IS NULL OR NR_PONT_PRFL_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS END AS NR_PONT_ESS_FIM,
    CASE WHEN NR_PONT_CONC_NAO_ESS IS NULL OR NR_PONT_ORC_NAO_ESS IS NULL OR NR_PONT_PRFL_NAO_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS END AS NR_PONT_NAO_ESS_FIM,
    CASE WHEN NR_PONT_CONC_FUT IS NULL OR NR_PONT_ORC_FUT IS NULL OR NR_PONT_PRFL_FUT IS NULL THEN NULL ELSE NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT END AS NR_PONT_FUT_FIM,
    CASE WHEN NR_PONT_CONC_OBR IS NULL OR NR_PONT_ORC_OBR IS NULL OR NR_PONT_PRFL_OBR IS NULL THEN NULL ELSE NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR END AS NR_PONT_OBR_FIM
FROM radar_motor_pontos
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_motor_resultado AS
WITH x AS (
    SELECT *,
        CASE WHEN NR_PONT_IND_FIM IS NOT NULL AND NR_PONT_ESS_FIM IS NOT NULL AND NR_PONT_NAO_ESS_FIM IS NOT NULL AND NR_PONT_FUT_FIM IS NOT NULL AND NR_PONT_OBR_FIM IS NOT NULL THEN 'S' ELSE 'N' END AS FL_PONTUACAO_COMPLETA
    FROM radar_motor_finais
), y AS (
    SELECT *,
        CASE WHEN FL_PONTUACAO_COMPLETA='S' THEN greatest(NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM) END AS NR_PONT_MAX
    FROM x
), z AS (
    SELECT *,
        CASE WHEN FL_PONTUACAO_COMPLETA='S' THEN
            CAST((CASE WHEN NR_PONT_IND_FIM=NR_PONT_MAX THEN 1 ELSE 0 END) +
                 (CASE WHEN NR_PONT_ESS_FIM=NR_PONT_MAX THEN 1 ELSE 0 END) +
                 (CASE WHEN NR_PONT_NAO_ESS_FIM=NR_PONT_MAX THEN 1 ELSE 0 END) +
                 (CASE WHEN NR_PONT_FUT_FIM=NR_PONT_MAX THEN 1 ELSE 0 END) +
                 (CASE WHEN NR_PONT_OBR_FIM=NR_PONT_MAX THEN 1 ELSE 0 END) AS INT)
        END AS QT_TEMAS_PONT_MAX
    FROM y
)
SELECT *,
    CASE WHEN FL_PONTUACAO_COMPLETA<>'S' THEN NULL WHEN QT_TEMAS_PONT_MAX>1 THEN 9 WHEN NR_PONT_IND_FIM=NR_PONT_MAX THEN 1 WHEN NR_PONT_ESS_FIM=NR_PONT_MAX THEN 2 WHEN NR_PONT_NAO_ESS_FIM=NR_PONT_MAX THEN 3 WHEN NR_PONT_FUT_FIM=NR_PONT_MAX THEN 4 ELSE 5 END AS CD_TEMA_VENCEDOR,
    CASE WHEN FL_PONTUACAO_COMPLETA<>'S' THEN NULL WHEN QT_TEMAS_PONT_MAX>1 THEN 'Empate' WHEN NR_PONT_IND_FIM=NR_PONT_MAX THEN 'Categorização dos Gastos' WHEN NR_PONT_ESS_FIM=NR_PONT_MAX THEN 'Gestão de Orçamento' WHEN NR_PONT_NAO_ESS_FIM=NR_PONT_MAX THEN 'Consumo Planejado' WHEN NR_PONT_FUT_FIM=NR_PONT_MAX THEN 'Formação de Reserva' ELSE 'Uso Consciente do Crédito' END AS TX_TEMA_VENCEDOR
FROM z
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_motor_pivot AS
SELECT CD_CLI,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN VL_RES_ORC END) AS VL_RES_ORC,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_SAI_ENT END) AS PC_SAI_ENT,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN CD_RES_ORC END) AS CD_RES_ORC,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN TX_RES_ORC END) AS TX_RES_ORC,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN CD_FAIXA_ORC END) AS CD_FAIXA_ORC,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN TX_STS_RES END) AS TX_STS_RES,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN TX_STS_FINAL END) AS TX_STS_FINAL,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_SAI_IND END) AS PC_SAI_IND,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_SAI_ESS END) AS PC_SAI_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_SAI_NAO_ESS END) AS PC_SAI_NAO_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_SAI_FUT END) AS PC_SAI_FUT,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_SAI_OBR END) AS PC_SAI_OBR,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_REF_IND END) AS PC_REF_IND,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_REF_ESS END) AS PC_REF_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_REF_NAO_ESS END) AS PC_REF_NAO_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_REF_FUT END) AS PC_REF_FUT,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN PC_REF_OBR END) AS PC_REF_OBR,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_CONC_IND END) AS NR_PONT_CONC_IND,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_CONC_ESS END) AS NR_PONT_CONC_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_CONC_NAO_ESS END) AS NR_PONT_CONC_NAO_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_CONC_FUT END) AS NR_PONT_CONC_FUT,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_CONC_OBR END) AS NR_PONT_CONC_OBR,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_ORC_IND END) AS NR_PONT_ORC_IND,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_ORC_ESS END) AS NR_PONT_ORC_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_ORC_NAO_ESS END) AS NR_PONT_ORC_NAO_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_ORC_FUT END) AS NR_PONT_ORC_FUT,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_ORC_OBR END) AS NR_PONT_ORC_OBR,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_PRFL_IND END) AS NR_PONT_PRFL_IND,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_PRFL_ESS END) AS NR_PONT_PRFL_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_PRFL_NAO_ESS END) AS NR_PONT_PRFL_NAO_ESS,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_PRFL_FUT END) AS NR_PONT_PRFL_FUT,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_PRFL_OBR END) AS NR_PONT_PRFL_OBR,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_IND_FIM END) AS NR_PONT_IND_FIM,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_ESS_FIM END) AS NR_PONT_ESS_FIM,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_NAO_ESS_FIM END) AS NR_PONT_NAO_ESS_FIM,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_FUT_FIM END) AS NR_PONT_FUT_FIM,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_OBR_FIM END) AS NR_PONT_OBR_FIM,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN FL_PONTUACAO_COMPLETA END) AS FL_PONTUACAO_COMPLETA,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN NR_PONT_MAX END) AS NR_PONT_MAX,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN QT_TEMAS_PONT_MAX END) AS QT_TEMAS_PONT_MAX,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN CD_TEMA_VENCEDOR END) AS CD_TEMA_VENCEDOR,
    MAX(CASE WHEN CENARIO='OFICIAL' THEN TX_TEMA_VENCEDOR END) AS TX_TEMA_VENCEDOR,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN VL_RES_ORC END) AS VL_RES_ORC_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN PC_SAI_ENT END) AS PC_SAI_ENT_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN CD_RES_ORC END) AS CD_RES_ORC_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN TX_RES_ORC END) AS TX_RES_ORC_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN CD_FAIXA_ORC END) AS CD_FAIXA_ORC_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN TX_STS_RES END) AS TX_STS_RES_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN TX_STS_FINAL END) AS TX_STS_FINAL_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN PC_SAI_IND END) AS PC_SAI_IND_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN PC_SAI_ESS END) AS PC_SAI_ESS_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN PC_SAI_NAO_ESS END) AS PC_SAI_NAO_ESS_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN PC_SAI_FUT END) AS PC_SAI_FUT_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN PC_SAI_OBR END) AS PC_SAI_OBR_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_CONC_IND END) AS NR_PONT_CONC_IND_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_CONC_ESS END) AS NR_PONT_CONC_ESS_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_CONC_NAO_ESS END) AS NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_CONC_FUT END) AS NR_PONT_CONC_FUT_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_CONC_OBR END) AS NR_PONT_CONC_OBR_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_ORC_ESS END) AS NR_PONT_ORC_ESS_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_ORC_NAO_ESS END) AS NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_ORC_FUT END) AS NR_PONT_ORC_FUT_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_ORC_OBR END) AS NR_PONT_ORC_OBR_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_IND_FIM END) AS NR_PONT_IND_FIM_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_ESS_FIM END) AS NR_PONT_ESS_FIM_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_NAO_ESS_FIM END) AS NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_FUT_FIM END) AS NR_PONT_FUT_FIM_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_OBR_FIM END) AS NR_PONT_OBR_FIM_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN FL_PONTUACAO_COMPLETA END) AS FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN NR_PONT_MAX END) AS NR_PONT_MAX_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN QT_TEMAS_PONT_MAX END) AS QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN CD_TEMA_VENCEDOR END) AS CD_TEMA_VENCEDOR_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='RENDA_PRESUMIDA' THEN TX_TEMA_VENCEDOR END) AS TX_TEMA_VENCEDOR_RENDA_PRESUMIDA,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN VL_RES_ORC END) AS VL_RES_ORC_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN PC_SAI_ENT END) AS PC_SAI_ENT_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN CD_RES_ORC END) AS CD_RES_ORC_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN TX_RES_ORC END) AS TX_RES_ORC_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN CD_FAIXA_ORC END) AS CD_FAIXA_ORC_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN TX_STS_RES END) AS TX_STS_RES_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN TX_STS_FINAL END) AS TX_STS_FINAL_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN PC_SAI_IND END) AS PC_SAI_IND_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN PC_SAI_ESS END) AS PC_SAI_ESS_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN PC_SAI_NAO_ESS END) AS PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN PC_SAI_FUT END) AS PC_SAI_FUT_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN PC_SAI_OBR END) AS PC_SAI_OBR_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_CONC_IND END) AS NR_PONT_CONC_IND_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_CONC_ESS END) AS NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_CONC_NAO_ESS END) AS NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_CONC_FUT END) AS NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_CONC_OBR END) AS NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_ORC_ESS END) AS NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_ORC_NAO_ESS END) AS NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_ORC_FUT END) AS NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_ORC_OBR END) AS NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_IND_FIM END) AS NR_PONT_IND_FIM_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_ESS_FIM END) AS NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_NAO_ESS_FIM END) AS NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_FUT_FIM END) AS NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_OBR_FIM END) AS NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN FL_PONTUACAO_COMPLETA END) AS FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN NR_PONT_MAX END) AS NR_PONT_MAX_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN QT_TEMAS_PONT_MAX END) AS QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN CD_TEMA_VENCEDOR END) AS CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS,
    MAX(CASE WHEN CENARIO='ENTRADAS_REALIZADAS' THEN TX_TEMA_VENCEDOR END) AS TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS
FROM radar_motor_resultado
GROUP BY CD_CLI
""")
spark.sql('CACHE TABLE radar_motor_pivot')
print('Motor clientes:', spark.sql('SELECT COUNT(*) N FROM radar_motor_pivot').first()['N'])
metricas_execucao['MOTOR'] = {'segundos': round(time.perf_counter() - t0_etapa, 3)}
print('[TEMPO MOTOR]', metricas_execucao['MOTOR']['segundos'], 's')


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
# Resultado final — Projetar explicitamente o contrato de 142 colunas.
spark.sql("""
CREATE OR REPLACE TEMP VIEW radar_resultado_final AS
SELECT
    CAST(b.CD_CLI AS INT) AS CD_CLI,
    CAST(b.DT_EXEA AS DATE) AS DT_EXEA,
    CAST(b.DT_MES_EXEA AS DATE) AS DT_MES_EXEA,
    CAST(b.TS_INCL_TRAN_REF AS TIMESTAMP) AS TS_INCL_TRAN_REF,
    CAST(b.FL_CPF_UNICO AS STRING) AS FL_CPF_UNICO,
    CAST(b.CD_CPF AS DECIMAL(14,0)) AS CD_CPF,
    CAST(b.FL_CONTA_ELEGIVEL_UNICA AS STRING) AS FL_CONTA_ELEGIVEL_UNICA,
    CAST(b.TS_DD_INC_MM_CLC_BLC_REF AS TIMESTAMP) AS TS_DD_INC_MM_CLC_BLC_REF,
    CAST(b.DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC,
    CAST(b.DD_INC_MM_CLC_BLC_FALLBACK AS SMALLINT) AS DD_INC_MM_CLC_BLC_FALLBACK,
    CAST(b.DT_REN_PRES_REF AS DATE) AS DT_REN_PRES_REF,
    CAST(b.VL_REN_PRES AS DECIMAL(17,2)) AS VL_REN_PRES,
    CAST(b.DT_REF_PRFL AS DATE) AS DT_REF_PRFL,
    CAST(b.CD_MAC_PRFL_CLI AS INT) AS CD_MAC_PRFL_CLI,
    CAST(b.NM_MAC_PRFL_CLI AS STRING) AS NM_MAC_PRFL_CLI,
    CAST(b.CD_MIC_PRFL_CLI AS INT) AS CD_MIC_PRFL_CLI,
    CAST(b.NM_MIC_PRFL_CLI AS STRING) AS NM_MIC_PRFL_CLI,
    CAST(b.DT_REF_INI AS DATE) AS DT_REF_INI,
    CAST(b.DT_REF_FIM AS DATE) AS DT_REF_FIM,
    CAST(b.FL_SOMENTE_BRL AS STRING) AS FL_SOMENTE_BRL,
    CAST(b.FL_TEM_MOV_AGRO AS STRING) AS FL_TEM_MOV_AGRO,
    CAST(b.QT_TRANS_TOTAL AS BIGINT) AS QT_TRANS_TOTAL,
    CAST(b.QT_TRANS_ENT AS BIGINT) AS QT_TRANS_ENT,
    CAST(b.QT_TRANS_SAI AS BIGINT) AS QT_TRANS_SAI,
    CAST(b.VL_TRANS_ENT AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(b.VL_TRANS_SAI AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(b.VL_ENT_REN AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(b.VL_ENT_EST AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(b.VL_ENT_RESG AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(b.VL_ENT_OUT AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(b.VL_ENT_CRED AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(b.VL_ENT_TOTAL AS DECIMAL(25,2)) AS VL_ENT_TOTAL,
    CAST(b.VL_SAI_IND AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(b.VL_SAI_ESS AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(b.VL_SAI_NAO_ESS AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(b.VL_SAI_FUT AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(b.VL_SAI_OBR AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(b.VL_SAI_TOTAL AS DECIMAL(25,2)) AS VL_SAI_TOTAL,
    CAST(m.VL_RES_ORC AS DECIMAL(25,2)) AS VL_RES_ORC,
    CAST(m.PC_SAI_ENT AS DECIMAL(9,6)) AS PC_SAI_ENT,
    CAST(m.CD_RES_ORC AS INT) AS CD_RES_ORC,
    CAST(m.TX_RES_ORC AS STRING) AS TX_RES_ORC,
    CAST(m.CD_FAIXA_ORC AS INT) AS CD_FAIXA_ORC,
    CAST(m.TX_STS_RES AS STRING) AS TX_STS_RES,
    CAST(m.TX_STS_FINAL AS STRING) AS TX_STS_FINAL,
    CAST(m.PC_SAI_IND AS DECIMAL(9,6)) AS PC_SAI_IND,
    CAST(m.PC_SAI_ESS AS DECIMAL(9,6)) AS PC_SAI_ESS,
    CAST(m.PC_SAI_NAO_ESS AS DECIMAL(9,6)) AS PC_SAI_NAO_ESS,
    CAST(m.PC_SAI_FUT AS DECIMAL(9,6)) AS PC_SAI_FUT,
    CAST(m.PC_SAI_OBR AS DECIMAL(9,6)) AS PC_SAI_OBR,
    CAST(m.PC_REF_IND AS DECIMAL(9,6)) AS PC_REF_IND,
    CAST(m.PC_REF_ESS AS DECIMAL(9,6)) AS PC_REF_ESS,
    CAST(m.PC_REF_NAO_ESS AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
    CAST(m.PC_REF_FUT AS DECIMAL(9,6)) AS PC_REF_FUT,
    CAST(m.PC_REF_OBR AS DECIMAL(9,6)) AS PC_REF_OBR,
    CAST(m.NR_PONT_CONC_IND AS INT) AS NR_PONT_CONC_IND,
    CAST(m.NR_PONT_CONC_ESS AS INT) AS NR_PONT_CONC_ESS,
    CAST(m.NR_PONT_CONC_NAO_ESS AS INT) AS NR_PONT_CONC_NAO_ESS,
    CAST(m.NR_PONT_CONC_FUT AS INT) AS NR_PONT_CONC_FUT,
    CAST(m.NR_PONT_CONC_OBR AS INT) AS NR_PONT_CONC_OBR,
    CAST(m.NR_PONT_ORC_IND AS INT) AS NR_PONT_ORC_IND,
    CAST(m.NR_PONT_ORC_ESS AS INT) AS NR_PONT_ORC_ESS,
    CAST(m.NR_PONT_ORC_NAO_ESS AS INT) AS NR_PONT_ORC_NAO_ESS,
    CAST(m.NR_PONT_ORC_FUT AS INT) AS NR_PONT_ORC_FUT,
    CAST(m.NR_PONT_ORC_OBR AS INT) AS NR_PONT_ORC_OBR,
    CAST(m.NR_PONT_PRFL_IND AS INT) AS NR_PONT_PRFL_IND,
    CAST(m.NR_PONT_PRFL_ESS AS INT) AS NR_PONT_PRFL_ESS,
    CAST(m.NR_PONT_PRFL_NAO_ESS AS INT) AS NR_PONT_PRFL_NAO_ESS,
    CAST(m.NR_PONT_PRFL_FUT AS INT) AS NR_PONT_PRFL_FUT,
    CAST(m.NR_PONT_PRFL_OBR AS INT) AS NR_PONT_PRFL_OBR,
    CAST(m.NR_PONT_IND_FIM AS INT) AS NR_PONT_IND_FIM,
    CAST(m.NR_PONT_ESS_FIM AS INT) AS NR_PONT_ESS_FIM,
    CAST(m.NR_PONT_NAO_ESS_FIM AS INT) AS NR_PONT_NAO_ESS_FIM,
    CAST(m.NR_PONT_FUT_FIM AS INT) AS NR_PONT_FUT_FIM,
    CAST(m.NR_PONT_OBR_FIM AS INT) AS NR_PONT_OBR_FIM,
    CAST(m.FL_PONTUACAO_COMPLETA AS STRING) AS FL_PONTUACAO_COMPLETA,
    CAST(m.NR_PONT_MAX AS INT) AS NR_PONT_MAX,
    CAST(m.QT_TEMAS_PONT_MAX AS INT) AS QT_TEMAS_PONT_MAX,
    CAST(m.CD_TEMA_VENCEDOR AS INT) AS CD_TEMA_VENCEDOR,
    CAST(m.TX_TEMA_VENCEDOR AS STRING) AS TX_TEMA_VENCEDOR,
    CAST(m.VL_RES_ORC_RENDA_PRESUMIDA AS DECIMAL(25,2)) AS VL_RES_ORC_RENDA_PRESUMIDA,
    CAST(m.PC_SAI_ENT_RENDA_PRESUMIDA AS DECIMAL(9,6)) AS PC_SAI_ENT_RENDA_PRESUMIDA,
    CAST(m.CD_RES_ORC_RENDA_PRESUMIDA AS INT) AS CD_RES_ORC_RENDA_PRESUMIDA,
    CAST(m.TX_RES_ORC_RENDA_PRESUMIDA AS STRING) AS TX_RES_ORC_RENDA_PRESUMIDA,
    CAST(m.CD_FAIXA_ORC_RENDA_PRESUMIDA AS INT) AS CD_FAIXA_ORC_RENDA_PRESUMIDA,
    CAST(m.TX_STS_RES_RENDA_PRESUMIDA AS STRING) AS TX_STS_RES_RENDA_PRESUMIDA,
    CAST(m.TX_STS_FINAL_RENDA_PRESUMIDA AS STRING) AS TX_STS_FINAL_RENDA_PRESUMIDA,
    CAST(m.PC_SAI_IND_RENDA_PRESUMIDA AS DECIMAL(9,6)) AS PC_SAI_IND_RENDA_PRESUMIDA,
    CAST(m.PC_SAI_ESS_RENDA_PRESUMIDA AS DECIMAL(9,6)) AS PC_SAI_ESS_RENDA_PRESUMIDA,
    CAST(m.PC_SAI_NAO_ESS_RENDA_PRESUMIDA AS DECIMAL(9,6)) AS PC_SAI_NAO_ESS_RENDA_PRESUMIDA,
    CAST(m.PC_SAI_FUT_RENDA_PRESUMIDA AS DECIMAL(9,6)) AS PC_SAI_FUT_RENDA_PRESUMIDA,
    CAST(m.PC_SAI_OBR_RENDA_PRESUMIDA AS DECIMAL(9,6)) AS PC_SAI_OBR_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_CONC_IND_RENDA_PRESUMIDA AS INT) AS NR_PONT_CONC_IND_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_CONC_ESS_RENDA_PRESUMIDA AS INT) AS NR_PONT_CONC_ESS_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA AS INT) AS NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_CONC_FUT_RENDA_PRESUMIDA AS INT) AS NR_PONT_CONC_FUT_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_CONC_OBR_RENDA_PRESUMIDA AS INT) AS NR_PONT_CONC_OBR_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_ORC_ESS_RENDA_PRESUMIDA AS INT) AS NR_PONT_ORC_ESS_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA AS INT) AS NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_ORC_FUT_RENDA_PRESUMIDA AS INT) AS NR_PONT_ORC_FUT_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_ORC_OBR_RENDA_PRESUMIDA AS INT) AS NR_PONT_ORC_OBR_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_IND_FIM_RENDA_PRESUMIDA AS INT) AS NR_PONT_IND_FIM_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_ESS_FIM_RENDA_PRESUMIDA AS INT) AS NR_PONT_ESS_FIM_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA AS INT) AS NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_FUT_FIM_RENDA_PRESUMIDA AS INT) AS NR_PONT_FUT_FIM_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_OBR_FIM_RENDA_PRESUMIDA AS INT) AS NR_PONT_OBR_FIM_RENDA_PRESUMIDA,
    CAST(m.FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA AS STRING) AS FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA,
    CAST(m.NR_PONT_MAX_RENDA_PRESUMIDA AS INT) AS NR_PONT_MAX_RENDA_PRESUMIDA,
    CAST(m.QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA AS INT) AS QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA,
    CAST(m.CD_TEMA_VENCEDOR_RENDA_PRESUMIDA AS INT) AS CD_TEMA_VENCEDOR_RENDA_PRESUMIDA,
    CAST(m.TX_TEMA_VENCEDOR_RENDA_PRESUMIDA AS STRING) AS TX_TEMA_VENCEDOR_RENDA_PRESUMIDA,
    CAST(m.VL_RES_ORC_ENTRADAS_REALIZADAS AS DECIMAL(25,2)) AS VL_RES_ORC_ENTRADAS_REALIZADAS,
    CAST(m.PC_SAI_ENT_ENTRADAS_REALIZADAS AS DECIMAL(9,6)) AS PC_SAI_ENT_ENTRADAS_REALIZADAS,
    CAST(m.CD_RES_ORC_ENTRADAS_REALIZADAS AS INT) AS CD_RES_ORC_ENTRADAS_REALIZADAS,
    CAST(m.TX_RES_ORC_ENTRADAS_REALIZADAS AS STRING) AS TX_RES_ORC_ENTRADAS_REALIZADAS,
    CAST(m.CD_FAIXA_ORC_ENTRADAS_REALIZADAS AS INT) AS CD_FAIXA_ORC_ENTRADAS_REALIZADAS,
    CAST(m.TX_STS_RES_ENTRADAS_REALIZADAS AS STRING) AS TX_STS_RES_ENTRADAS_REALIZADAS,
    CAST(m.TX_STS_FINAL_ENTRADAS_REALIZADAS AS STRING) AS TX_STS_FINAL_ENTRADAS_REALIZADAS,
    CAST(m.PC_SAI_IND_ENTRADAS_REALIZADAS AS DECIMAL(9,6)) AS PC_SAI_IND_ENTRADAS_REALIZADAS,
    CAST(m.PC_SAI_ESS_ENTRADAS_REALIZADAS AS DECIMAL(9,6)) AS PC_SAI_ESS_ENTRADAS_REALIZADAS,
    CAST(m.PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS AS DECIMAL(9,6)) AS PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS,
    CAST(m.PC_SAI_FUT_ENTRADAS_REALIZADAS AS DECIMAL(9,6)) AS PC_SAI_FUT_ENTRADAS_REALIZADAS,
    CAST(m.PC_SAI_OBR_ENTRADAS_REALIZADAS AS DECIMAL(9,6)) AS PC_SAI_OBR_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_CONC_IND_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_CONC_IND_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_IND_FIM_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_IND_FIM_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS,
    CAST(m.FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS AS STRING) AS FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS,
    CAST(m.NR_PONT_MAX_ENTRADAS_REALIZADAS AS INT) AS NR_PONT_MAX_ENTRADAS_REALIZADAS,
    CAST(m.QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS AS INT) AS QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS,
    CAST(m.CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS AS INT) AS CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS,
    CAST(m.TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS AS STRING) AS TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS
FROM radar_base b
LEFT JOIN radar_motor_pivot m ON b.CD_CLI=m.CD_CLI
""")
spark.sql('CACHE TABLE radar_resultado_final')
g=spark.sql('SELECT COUNT(*) N, COUNT(DISTINCT CD_CLI) U, SUM(CASE WHEN CD_CLI IS NULL THEN 1 ELSE 0 END) Z FROM radar_resultado_final').first()
if g['N']==0 or g['N']!=g['U'] or g['Z']!=0: raise RuntimeError(f'Resultado final inválido: {g}')
if len(spark.table('radar_resultado_final').columns) != 142: raise RuntimeError('Resultado final deve possuir 142 colunas.')
print('Resultado final:', g['N'], 'clientes | 142 colunas')
metricas_execucao['RESULTADO_FINAL'] = {'segundos': round(time.perf_counter() - t0_etapa, 3)}
print('[TEMPO RESULTADO_FINAL]', metricas_execucao['RESULTADO_FINAL']['segundos'], 's')


## 10. Gates finais


In [ ]:
%%time
%%spark
# Gates — Preservar o público e impedir publicação de resultado inconsistente.
t0_etapa = time.perf_counter()
g=spark.sql("""
SELECT
  (SELECT COUNT(*) FROM radar_publico) PUBLICO,
  (SELECT COUNT(*) FROM radar_resultado_final) RESULTADO,
  (SELECT COUNT(*) FROM radar_publico p LEFT ANTI JOIN radar_resultado_final r ON p.CD_CLI=r.CD_CLI) AUSENTES,
  (SELECT COUNT(*) FROM radar_resultado_final r LEFT ANTI JOIN radar_publico p ON r.CD_CLI=p.CD_CLI) EXTRAS
""").first()
if g['PUBLICO']!=g['RESULTADO'] or g['AUSENTES'] or g['EXTRAS']:
    raise RuntimeError(f'Grão público/resultado inválido: {g}')
print('Gates finais: OK')
metricas_execucao['GATES_FINAIS'] = {'segundos': round(time.perf_counter()-t0_etapa,3), 'publico': int(g['PUBLICO']), 'resultado': int(g['RESULTADO'])}
print('[GATES]', metricas_execucao['GATES_FINAIS'])


## 11. Resumo da execução calculada


In [ ]:
%%time
%%spark
print('=' * 100)
print('RESUMO V9 R4 — EXECUÇÃO CALCULADA')
print('=' * 100)
for etapa, info in metricas_execucao.items():
    if etapa == 'Q4_LEITURA' and isinstance(info, dict) and 'por_referencia' in info:
        resumo = {k:v for k,v in info.items() if k != 'por_referencia'}
        print(etapa, resumo)
        for ref in info['por_referencia']:
            print('   ', ref)
    else:
        print(etapa, info)

print('-' * 100)
print('PUBLICAR_RESULTADO =', PUBLICAR_RESULTADO)
print('Resultado final:', spark.sql('SELECT COUNT(*) N FROM radar_resultado_final').first()['N'])
print('Colunas finais:', len(spark.table('radar_resultado_final').columns))
print('=' * 100)


## 12. Publicação opcional

A publicação continua sendo o procedimento excepcional definido para este pipeline:

`resultado validado → DROP → DDL literal → INSERT fotografia completa → readback`.

Durante testes longos, `PUBLICAR_RESULTADO` fica `False` por padrão. Se estiver `False`, as células abaixo não alteram a tabela.


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
# DDL oficial — Recriar a tabela somente após o resultado estar validado.
DDL_ANA_RADAR_FIN_CLI = """
CREATE TABLE ANA_RADAR_FIN_CLI (

    -- ============================================================
    -- IDENTIFICAÇÃO, REFERÊNCIAS E CONTEXTO
    -- ============================================================

    CD_CLI INT
        COMMENT 'Código identificador do cliente processado pelo Radar',

    DT_EXEA DATE
        COMMENT 'Data de execução do cálculo do Radar',

    DT_MES_EXEA DATE
        COMMENT 'Primeiro dia do mês correspondente à data de execução',

    TS_INCL_TRAN_REF TIMESTAMP
        COMMENT 'Maior timestamp de inclusão de transação utilizado como referência para formação da janela financeira',

    FL_CPF_UNICO STRING
        COMMENT 'Indica se existe exatamente um CPF distinto associado ao cliente no público analisado: S=sim; N=não',

    CD_CPF DECIMAL(14,0)
        COMMENT 'Código do CPF único associado ao cliente; nulo quando não existe unicidade',

    FL_CONTA_ELEGIVEL_UNICA STRING
        COMMENT 'Indica se existe exatamente uma conta elegível para determinação do ciclo financeiro: S=sim; N=não',

    TS_DD_INC_MM_CLC_BLC_REF TIMESTAMP
        COMMENT 'Timestamp de referência do registro de ciclo financeiro selecionado',

    DD_INC_MM_CLC_BLC SMALLINT
        COMMENT 'Dia de início do ciclo financeiro obtido da fonte',

    DD_INC_MM_CLC_BLC_FALLBACK SMALLINT
        COMMENT 'Dia de ciclo efetivamente utilizado após aplicação da regra de fallback',

    DT_REN_PRES_REF DATE
        COMMENT 'Data de referência da renda presumida selecionada',

    VL_REN_PRES DECIMAL(17,2)
        COMMENT 'Valor da renda presumida utilizada pelo Radar',

    DT_REF_PRFL DATE
        COMMENT 'Data de referência do perfil financeiro selecionado',

    CD_MAC_PRFL_CLI INT
        COMMENT 'Código do macroperfil financeiro recebido da fonte; o notebook usa 1, 2 e 3 na pontuação, sem definir rótulos negociais próprios',

    NM_MAC_PRFL_CLI STRING
        COMMENT 'Nome do macroperfil financeiro recebido da fonte',

    CD_MIC_PRFL_CLI INT
        COMMENT 'Código do microperfil financeiro recebido da fonte; o motor atual não cria um de-para negocial próprio para este código',

    NM_MIC_PRFL_CLI STRING
        COMMENT 'Nome do microperfil financeiro recebido da fonte',

    DT_REF_INI DATE
        COMMENT 'Data inicial da janela financeira analisada',

    DT_REF_FIM DATE
        COMMENT 'Data final da janela financeira analisada',

    FL_SOMENTE_BRL STRING
        COMMENT 'Calculado sobre os movimentos efetivos: S quando a única moeda distinta é BRL; N nos demais casos; NULL quando não há movimentos efetivos',

    FL_TEM_MOV_AGRO STRING
        COMMENT 'Indica movimentação agro entre os movimentos efetivos BRL: S=possui movimento agro; N=não possui; NULL=sem movimentos BRL',


    -- ============================================================
    -- MOVIMENTAÇÃO
    -- ============================================================

    QT_TRANS_TOTAL BIGINT
        COMMENT 'Quantidade total de movimentos efetivos BRL considerados na janela',

    QT_TRANS_ENT BIGINT
        COMMENT 'Quantidade de movimentos efetivos de entrada em BRL',

    QT_TRANS_SAI BIGINT
        COMMENT 'Quantidade de movimentos efetivos de saída em BRL',

    VL_TRANS_ENT DECIMAL(25,2)
        COMMENT 'Valor das entradas participantes do cálculo; no contrato atual corresponde ao VL_ENT_TOTAL',

    VL_TRANS_SAI DECIMAL(25,2)
        COMMENT 'Valor das saídas participantes do cálculo; no contrato atual corresponde ao VL_SAI_TOTAL',


    -- ============================================================
    -- ENTRADAS
    -- ============================================================

    VL_ENT_REN DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Renda, classe 1',

    VL_ENT_EST DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Estorno, classe 2',

    VL_ENT_RESG DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Resgate, classe 3',

    VL_ENT_OUT DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Outras Entradas, classe 0',

    VL_ENT_CRED DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Crédito, classe 4',

    VL_ENT_TOTAL DECIMAL(25,2)
        COMMENT 'Valor total das entradas participantes do orçamento',


    -- ============================================================
    -- SAÍDAS
    -- ============================================================

    VL_SAI_IND DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Indeterminadas, classe 5',

    VL_SAI_ESS DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Essenciais, classe 6',

    VL_SAI_NAO_ESS DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Não Essenciais, classe 7',

    VL_SAI_FUT DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Futuro, classe 8',

    VL_SAI_OBR DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Obrigações, classe 9',

    VL_SAI_TOTAL DECIMAL(25,2)
        COMMENT 'Valor total das saídas participantes do orçamento',


    -- ============================================================
    -- RESULTADO ORÇAMENTÁRIO OFICIAL
    -- ============================================================

    VL_RES_ORC DECIMAL(25,2)
        COMMENT 'Resultado orçamentário calculado pela diferença entre VL_ENT_TOTAL e VL_SAI_TOTAL',

    PC_SAI_ENT DECIMAL(9,6)
        COMMENT 'Relação entre o total de saídas e a base de entrada utilizada no orçamento oficial',

    CD_RES_ORC INT
        COMMENT 'Código do resultado orçamentário: 0=Neutro; 1=Superavitário; 2=Deficitário',

    TX_RES_ORC STRING
        COMMENT 'Descrição do resultado orçamentário correspondente ao CD_RES_ORC',

    CD_FAIXA_ORC INT
        COMMENT 'Código da faixa orçamentária: 0=Neutro; 1=Deficitário Moderado; 2=Deficitário Acentuado; 3=Superavitário Moderado; 4=Superavitário Acentuado',

    TX_STS_RES STRING
        COMMENT 'Intensidade do resultado orçamentário: Moderado ou Acentuado; NULL para resultado Neutro ou não calculado',

    TX_STS_FINAL STRING
        COMMENT 'Descrição final do resultado: Neutro, Deficitário Moderado, Deficitário Acentuado, Superavitário Moderado ou Superavitário Acentuado',


    -- ============================================================
    -- PERCENTUAIS OFICIAIS
    -- ============================================================

    PC_SAI_IND DECIMAL(9,6)
        COMMENT 'Proporção das saídas indeterminadas em relação à renda presumida no resultado oficial',

    PC_SAI_ESS DECIMAL(9,6)
        COMMENT 'Proporção das saídas essenciais em relação à renda presumida no resultado oficial',

    PC_SAI_NAO_ESS DECIMAL(9,6)
        COMMENT 'Proporção das saídas não essenciais em relação à renda presumida no resultado oficial',

    PC_SAI_FUT DECIMAL(9,6)
        COMMENT 'Proporção das saídas para futuro em relação à renda presumida no resultado oficial',

    PC_SAI_OBR DECIMAL(9,6)
        COMMENT 'Proporção das saídas de obrigações em relação à renda presumida no resultado oficial',

    PC_REF_IND DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Indeterminada: 0.750000 corresponde a 75%',

    PC_REF_ESS DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Essencial: 0.500000 corresponde a 50%',

    PC_REF_NAO_ESS DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Não Essencial: 0.300000 corresponde a 30%',

    PC_REF_FUT DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Futuro: 0.200000 corresponde a 20%',

    PC_REF_OBR DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Obrigações: 0.300000 corresponde a 30%',


    -- ============================================================
    -- PONTUAÇÃO DE CONCENTRAÇÃO
    -- ============================================================

    NR_PONT_CONC_IND INT
        COMMENT 'Pontuação de concentração da categoria Indeterminada; valores produzidos pelo motor incluem 0 e 99',

    NR_PONT_CONC_ESS INT
        COMMENT 'Pontuação de concentração da categoria Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_CONC_NAO_ESS INT
        COMMENT 'Pontuação de concentração da categoria Não Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_CONC_FUT INT
        COMMENT 'Pontuação de concentração da categoria Futuro; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_CONC_OBR INT
        COMMENT 'Pontuação de concentração da categoria Obrigações; valores produzidos pelo motor: 0, 1 ou 2',


    -- ============================================================
    -- PONTUAÇÃO ORÇAMENTÁRIA
    -- ============================================================

    NR_PONT_ORC_IND INT
        COMMENT 'Pontuação orçamentária da categoria Indeterminada',

    NR_PONT_ORC_ESS INT
        COMMENT 'Pontuação orçamentária da categoria Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_ORC_NAO_ESS INT
        COMMENT 'Pontuação orçamentária da categoria Não Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_ORC_FUT INT
        COMMENT 'Pontuação orçamentária da categoria Futuro; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_ORC_OBR INT
        COMMENT 'Pontuação orçamentária da categoria Obrigações; valores produzidos pelo motor: 0, 1 ou 2',


    -- ============================================================
    -- PONTUAÇÃO DE PERFIL
    -- ============================================================

    NR_PONT_PRFL_IND INT
        COMMENT 'Pontuação de perfil da categoria Indeterminada',

    NR_PONT_PRFL_ESS INT
        COMMENT 'Pontuação de perfil da categoria Essencial',

    NR_PONT_PRFL_NAO_ESS INT
        COMMENT 'Pontuação de perfil da categoria Não Essencial',

    NR_PONT_PRFL_FUT INT
        COMMENT 'Pontuação de perfil da categoria Futuro',

    NR_PONT_PRFL_OBR INT
        COMMENT 'Pontuação de perfil da categoria Obrigações',


    -- ============================================================
    -- PONTUAÇÃO FINAL E TEMA VENCEDOR
    -- ============================================================

    NR_PONT_IND_FIM INT
        COMMENT 'Pontuação final do tema 1, Categorização dos Gastos',

    NR_PONT_ESS_FIM INT
        COMMENT 'Pontuação final do tema 2, Gestão de Orçamento',

    NR_PONT_NAO_ESS_FIM INT
        COMMENT 'Pontuação final do tema 3, Consumo Planejado',

    NR_PONT_FUT_FIM INT
        COMMENT 'Pontuação final do tema 4, Formação de Reserva',

    NR_PONT_OBR_FIM INT
        COMMENT 'Pontuação final do tema 5, Uso Consciente do Crédito',

    FL_PONTUACAO_COMPLETA STRING
        COMMENT 'Indica se as cinco pontuações finais foram calculadas: S=completa; N=incompleta',

    NR_PONT_MAX INT
        COMMENT 'Maior pontuação final obtida entre os cinco temas',

    QT_TEMAS_PONT_MAX INT
        COMMENT 'Quantidade de temas que possuem a maior pontuação final; exemplo: 1 indica vencedor único e valor maior que 1 indica empate',

    CD_TEMA_VENCEDOR INT
        COMMENT 'Código do tema vencedor: 1=Categorização dos Gastos; 2=Gestão de Orçamento; 3=Consumo Planejado; 4=Formação de Reserva; 5=Uso Consciente do Crédito; 9=Empate; NULL=pontuação incompleta',

    TX_TEMA_VENCEDOR STRING
        COMMENT 'Descrição correspondente ao CD_TEMA_VENCEDOR',


    -- ============================================================
    -- CENÁRIO RENDA PRESUMIDA
    -- ============================================================

    VL_RES_ORC_RENDA_PRESUMIDA DECIMAL(25,2)
        COMMENT 'Resultado orçamentário recalculado no cenário RENDA_PRESUMIDA',

    PC_SAI_ENT_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Relação entre saídas e base utilizada no cenário RENDA_PRESUMIDA',

    CD_RES_ORC_RENDA_PRESUMIDA INT
        COMMENT 'Código do resultado orçamentário no cenário RENDA_PRESUMIDA: 0=Neutro; 1=Superavitário; 2=Deficitário',

    TX_RES_ORC_RENDA_PRESUMIDA STRING
        COMMENT 'Descrição do resultado orçamentário no cenário RENDA_PRESUMIDA',

    CD_FAIXA_ORC_RENDA_PRESUMIDA INT
        COMMENT 'Código da faixa orçamentária no cenário RENDA_PRESUMIDA: 0=Neutro; 1=Deficitário Moderado; 2=Deficitário Acentuado; 3=Superavitário Moderado; 4=Superavitário Acentuado',

    TX_STS_RES_RENDA_PRESUMIDA STRING
        COMMENT 'Intensidade do resultado orçamentário no cenário RENDA_PRESUMIDA',

    TX_STS_FINAL_RENDA_PRESUMIDA STRING
        COMMENT 'Descrição final do resultado orçamentário no cenário RENDA_PRESUMIDA',

    PC_SAI_IND_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas indeterminadas no cenário RENDA_PRESUMIDA',

    PC_SAI_ESS_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas essenciais no cenário RENDA_PRESUMIDA',

    PC_SAI_NAO_ESS_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas não essenciais no cenário RENDA_PRESUMIDA',

    PC_SAI_FUT_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas para futuro no cenário RENDA_PRESUMIDA',

    PC_SAI_OBR_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas de obrigações no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_IND_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Indeterminada no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Não Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_FUT_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Futuro no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_OBR_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Obrigações no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Não Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_FUT_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Futuro no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_OBR_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Obrigações no cenário RENDA_PRESUMIDA',

    NR_PONT_IND_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 1, Categorização dos Gastos, no cenário RENDA_PRESUMIDA',

    NR_PONT_ESS_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 2, Gestão de Orçamento, no cenário RENDA_PRESUMIDA',

    NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 3, Consumo Planejado, no cenário RENDA_PRESUMIDA',

    NR_PONT_FUT_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 4, Formação de Reserva, no cenário RENDA_PRESUMIDA',

    NR_PONT_OBR_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 5, Uso Consciente do Crédito, no cenário RENDA_PRESUMIDA',

    FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA STRING
        COMMENT 'Indica se as cinco pontuações foram calculadas no cenário RENDA_PRESUMIDA: S=completa; N=incompleta',

    NR_PONT_MAX_RENDA_PRESUMIDA INT
        COMMENT 'Maior pontuação final obtida no cenário RENDA_PRESUMIDA',

    QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA INT
        COMMENT 'Quantidade de temas com a maior pontuação no cenário RENDA_PRESUMIDA',

    CD_TEMA_VENCEDOR_RENDA_PRESUMIDA INT
        COMMENT 'Código do tema vencedor no cenário RENDA_PRESUMIDA: 1=Categorização dos Gastos; 2=Gestão de Orçamento; 3=Consumo Planejado; 4=Formação de Reserva; 5=Uso Consciente do Crédito; 9=Empate; NULL=pontuação incompleta',

    TX_TEMA_VENCEDOR_RENDA_PRESUMIDA STRING
        COMMENT 'Descrição correspondente ao tema vencedor no cenário RENDA_PRESUMIDA',


    -- ============================================================
    -- CENÁRIO ENTRADAS REALIZADAS
    -- ============================================================

    VL_RES_ORC_ENTRADAS_REALIZADAS DECIMAL(25,2)
        COMMENT 'Resultado orçamentário recalculado no cenário ENTRADAS_REALIZADAS',

    PC_SAI_ENT_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Relação entre saídas e base utilizada no cenário ENTRADAS_REALIZADAS',

    CD_RES_ORC_ENTRADAS_REALIZADAS INT
        COMMENT 'Código do resultado orçamentário no cenário ENTRADAS_REALIZADAS: 0=Neutro; 1=Superavitário; 2=Deficitário',

    TX_RES_ORC_ENTRADAS_REALIZADAS STRING
        COMMENT 'Descrição do resultado orçamentário no cenário ENTRADAS_REALIZADAS',

    CD_FAIXA_ORC_ENTRADAS_REALIZADAS INT
        COMMENT 'Código da faixa orçamentária no cenário ENTRADAS_REALIZADAS: 0=Neutro; 1=Deficitário Moderado; 2=Deficitário Acentuado; 3=Superavitário Moderado; 4=Superavitário Acentuado',

    TX_STS_RES_ENTRADAS_REALIZADAS STRING
        COMMENT 'Intensidade do resultado orçamentário no cenário ENTRADAS_REALIZADAS',

    TX_STS_FINAL_ENTRADAS_REALIZADAS STRING
        COMMENT 'Descrição final do resultado orçamentário no cenário ENTRADAS_REALIZADAS',

    PC_SAI_IND_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas indeterminadas no cenário ENTRADAS_REALIZADAS',

    PC_SAI_ESS_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas essenciais no cenário ENTRADAS_REALIZADAS',

    PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas não essenciais no cenário ENTRADAS_REALIZADAS',

    PC_SAI_FUT_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas para futuro no cenário ENTRADAS_REALIZADAS',

    PC_SAI_OBR_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas de obrigações no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_IND_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Indeterminada no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Não Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Futuro no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Obrigações no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Não Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Futuro no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Obrigações no cenário ENTRADAS_REALIZADAS',

    NR_PONT_IND_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 1, Categorização dos Gastos, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 2, Gestão de Orçamento, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 3, Consumo Planejado, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 4, Formação de Reserva, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 5, Uso Consciente do Crédito, no cenário ENTRADAS_REALIZADAS',

    FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS STRING
        COMMENT 'Indica se as cinco pontuações foram calculadas no cenário ENTRADAS_REALIZADAS: S=completa; N=incompleta',

    NR_PONT_MAX_ENTRADAS_REALIZADAS INT
        COMMENT 'Maior pontuação final obtida no cenário ENTRADAS_REALIZADAS',

    QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS INT
        COMMENT 'Quantidade de temas com a maior pontuação no cenário ENTRADAS_REALIZADAS',

    CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS INT
        COMMENT 'Código do tema vencedor no cenário ENTRADAS_REALIZADAS: 1=Categorização dos Gastos; 2=Gestão de Orçamento; 3=Consumo Planejado; 4=Formação de Reserva; 5=Uso Consciente do Crédito; 9=Empate; NULL=pontuação incompleta',

    TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS STRING
        COMMENT 'Descrição correspondente ao tema vencedor no cenário ENTRADAS_REALIZADAS'

)
COMMENT 'Resultado negocial do Radar Financeiro por cliente, contendo resultado oficial e os dois cenários calculados atualmente'
STORED AS PARQUET;
"""
if PUBLICAR_RESULTADO:
    spark.catalog.setCurrentDatabase(DB_ATIVO)
    spark.sql('DROP TABLE IF EXISTS ANA_RADAR_FIN_CLI')
    spark.sql(DDL_ANA_RADAR_FIN_CLI)
    metricas_execucao['PUBLICACAO_DDL'] = {'segundos': round(time.perf_counter()-t0_etapa,3)}
    print('Tabela recriada em:', spark.catalog.currentDatabase())
    print('[PUBLICACAO DDL]', metricas_execucao['PUBLICACAO_DDL'])
else:
    print('PUBLICAR_RESULTADO=False — DROP/DDL não executados.')


In [ ]:
%%time
%%spark
t0_etapa = time.perf_counter()
if PUBLICAR_RESULTADO:
    # Publicação — Inserir a fotografia completa e conferir o readback.
    spark.sql("""
    INSERT INTO TABLE ANA_RADAR_FIN_CLI
    SELECT
        CD_CLI,
        DT_EXEA,
        DT_MES_EXEA,
        TS_INCL_TRAN_REF,
        FL_CPF_UNICO,
        CD_CPF,
        FL_CONTA_ELEGIVEL_UNICA,
        TS_DD_INC_MM_CLC_BLC_REF,
        DD_INC_MM_CLC_BLC,
        DD_INC_MM_CLC_BLC_FALLBACK,
        DT_REN_PRES_REF,
        VL_REN_PRES,
        DT_REF_PRFL,
        CD_MAC_PRFL_CLI,
        NM_MAC_PRFL_CLI,
        CD_MIC_PRFL_CLI,
        NM_MIC_PRFL_CLI,
        DT_REF_INI,
        DT_REF_FIM,
        FL_SOMENTE_BRL,
        FL_TEM_MOV_AGRO,
        QT_TRANS_TOTAL,
        QT_TRANS_ENT,
        QT_TRANS_SAI,
        VL_TRANS_ENT,
        VL_TRANS_SAI,
        VL_ENT_REN,
        VL_ENT_EST,
        VL_ENT_RESG,
        VL_ENT_OUT,
        VL_ENT_CRED,
        VL_ENT_TOTAL,
        VL_SAI_IND,
        VL_SAI_ESS,
        VL_SAI_NAO_ESS,
        VL_SAI_FUT,
        VL_SAI_OBR,
        VL_SAI_TOTAL,
        VL_RES_ORC,
        PC_SAI_ENT,
        CD_RES_ORC,
        TX_RES_ORC,
        CD_FAIXA_ORC,
        TX_STS_RES,
        TX_STS_FINAL,
        PC_SAI_IND,
        PC_SAI_ESS,
        PC_SAI_NAO_ESS,
        PC_SAI_FUT,
        PC_SAI_OBR,
        PC_REF_IND,
        PC_REF_ESS,
        PC_REF_NAO_ESS,
        PC_REF_FUT,
        PC_REF_OBR,
        NR_PONT_CONC_IND,
        NR_PONT_CONC_ESS,
        NR_PONT_CONC_NAO_ESS,
        NR_PONT_CONC_FUT,
        NR_PONT_CONC_OBR,
        NR_PONT_ORC_IND,
        NR_PONT_ORC_ESS,
        NR_PONT_ORC_NAO_ESS,
        NR_PONT_ORC_FUT,
        NR_PONT_ORC_OBR,
        NR_PONT_PRFL_IND,
        NR_PONT_PRFL_ESS,
        NR_PONT_PRFL_NAO_ESS,
        NR_PONT_PRFL_FUT,
        NR_PONT_PRFL_OBR,
        NR_PONT_IND_FIM,
        NR_PONT_ESS_FIM,
        NR_PONT_NAO_ESS_FIM,
        NR_PONT_FUT_FIM,
        NR_PONT_OBR_FIM,
        FL_PONTUACAO_COMPLETA,
        NR_PONT_MAX,
        QT_TEMAS_PONT_MAX,
        CD_TEMA_VENCEDOR,
        TX_TEMA_VENCEDOR,
        VL_RES_ORC_RENDA_PRESUMIDA,
        PC_SAI_ENT_RENDA_PRESUMIDA,
        CD_RES_ORC_RENDA_PRESUMIDA,
        TX_RES_ORC_RENDA_PRESUMIDA,
        CD_FAIXA_ORC_RENDA_PRESUMIDA,
        TX_STS_RES_RENDA_PRESUMIDA,
        TX_STS_FINAL_RENDA_PRESUMIDA,
        PC_SAI_IND_RENDA_PRESUMIDA,
        PC_SAI_ESS_RENDA_PRESUMIDA,
        PC_SAI_NAO_ESS_RENDA_PRESUMIDA,
        PC_SAI_FUT_RENDA_PRESUMIDA,
        PC_SAI_OBR_RENDA_PRESUMIDA,
        NR_PONT_CONC_IND_RENDA_PRESUMIDA,
        NR_PONT_CONC_ESS_RENDA_PRESUMIDA,
        NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA,
        NR_PONT_CONC_FUT_RENDA_PRESUMIDA,
        NR_PONT_CONC_OBR_RENDA_PRESUMIDA,
        NR_PONT_ORC_ESS_RENDA_PRESUMIDA,
        NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA,
        NR_PONT_ORC_FUT_RENDA_PRESUMIDA,
        NR_PONT_ORC_OBR_RENDA_PRESUMIDA,
        NR_PONT_IND_FIM_RENDA_PRESUMIDA,
        NR_PONT_ESS_FIM_RENDA_PRESUMIDA,
        NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA,
        NR_PONT_FUT_FIM_RENDA_PRESUMIDA,
        NR_PONT_OBR_FIM_RENDA_PRESUMIDA,
        FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA,
        NR_PONT_MAX_RENDA_PRESUMIDA,
        QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA,
        CD_TEMA_VENCEDOR_RENDA_PRESUMIDA,
        TX_TEMA_VENCEDOR_RENDA_PRESUMIDA,
        VL_RES_ORC_ENTRADAS_REALIZADAS,
        PC_SAI_ENT_ENTRADAS_REALIZADAS,
        CD_RES_ORC_ENTRADAS_REALIZADAS,
        TX_RES_ORC_ENTRADAS_REALIZADAS,
        CD_FAIXA_ORC_ENTRADAS_REALIZADAS,
        TX_STS_RES_ENTRADAS_REALIZADAS,
        TX_STS_FINAL_ENTRADAS_REALIZADAS,
        PC_SAI_IND_ENTRADAS_REALIZADAS,
        PC_SAI_ESS_ENTRADAS_REALIZADAS,
        PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS,
        PC_SAI_FUT_ENTRADAS_REALIZADAS,
        PC_SAI_OBR_ENTRADAS_REALIZADAS,
        NR_PONT_CONC_IND_ENTRADAS_REALIZADAS,
        NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS,
        NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS,
        NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS,
        NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS,
        NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS,
        NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS,
        NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS,
        NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS,
        NR_PONT_IND_FIM_ENTRADAS_REALIZADAS,
        NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS,
        NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS,
        NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS,
        NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS,
        FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS,
        NR_PONT_MAX_ENTRADAS_REALIZADAS,
        QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS,
        CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS,
        TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS
    FROM radar_resultado_final
    """)

    g=spark.sql("""
    SELECT COUNT(*) N, COUNT(DISTINCT CD_CLI) U, SUM(CASE WHEN CD_CLI IS NULL THEN 1 ELSE 0 END) Z
    FROM ANA_RADAR_FIN_CLI
    """).first()
    orig=spark.sql('SELECT COUNT(*) N FROM radar_resultado_final').first()['N']
    if g['N']!=orig or g['N']!=g['U'] or g['Z']!=0:
        raise RuntimeError(f'Readback inválido: publicado={g}, esperado={orig}')

    schema_final=[(f.name,f.dataType.simpleString()) for f in spark.table('radar_resultado_final').schema.fields]
    schema_publicado=[(f.name,f.dataType.simpleString()) for f in spark.table('ANA_RADAR_FIN_CLI').schema.fields]
    if schema_final != schema_publicado:
        raise RuntimeError('Schema publicado difere do resultado final.')

    for tabela in ('radar_publico','radar_contexto','radar_base','radar_motor_pivot','radar_resultado_final','radar_pares_exatos','radar_pares_borda'):
        try: spark.sql(f'UNCACHE TABLE {tabela}')
        except Exception: pass
    print('Publicação concluída:', g['N'], 'clientes')
    metricas_execucao['PUBLICACAO_INSERT'] = {'segundos': round(time.perf_counter()-t0_etapa,3), 'clientes': int(g['N'])}
    print('[PUBLICACAO INSERT]', metricas_execucao['PUBLICACAO_INSERT'])
else:
    print('PUBLICAR_RESULTADO=False — INSERT/readback não executados.')


---

## Fim da V9 R2 completa de teste

Ao terminar, guardar principalmente:

- todos os blocos `[Q1]` a `[Q5]`;
- as linhas `[Q4 REF]`, que mostram a curva real de cobertura por `DT_REF`;
- tempos da reconciliação exata e de borda;
- volume de `radar_q5_raw`, `radar_movimentos`, residual e efetivos;
- resumo V9;
- Spark UI das etapas que dominarem o tempo, se possível.
